# Hypoxia Classifier

## Introduction

### Background 

Hypoxia, the oxygen shortage that develops when a tumour outgrows its blood supply, has serious clinical consequences. Cancer cells starved of oxygen often survive radiotherapy and chemotherapy that would otherwise kill them, and they spread more readily to other tissues. This resilience is driven largely by gene expression: a cell under low oxygen switches on a characteristic set of genes, leaving a recognisable signature in its transcriptome. Single-cell RNA sequencing records that signature for each cell individually, which raises the question of whether a cell's oxygen state can be inferred from its gene activity alone. Our data come from two breast cancer cell lines, MCF7 and HCC1806, each grown under normal and low-oxygen conditions and profiled with two sequencing technologies, SmartSeq and DropSeq. The two lines are different breast cancer subtypes: MCF7 is estrogen-receptor positive and luminal, HCC1806 triple-negative and basal-like and more aggressive. The contrast lets us test whether a hypoxia signature holds across subtypes or only within one.

**Aim**: train machine learning models that can tell, from gene expression alone, whether a single cell is hypoxic, and then to test how far that ability holds when a model meets a cell line or a sequencing technology it never saw during training

### Methods and materials

*to be continued*

### Data

*describe the 2 different sequencing methods*

## 0. Imports and setup

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, classification_report, ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
import pickle
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

#reproducability
RNG = np.random.default_rng(42)
RANDOM_STATE = 42

ROOT  = Path.cwd()
DATA  = ROOT / "Data"
SMART = DATA / "SmartSeq"
DROP  = DATA / "DropSeq"

for label, p in [("ROOT", ROOT), ("DATA", DATA), ("SMART", SMART), ("DROP", DROP)]:
    print(f"{label:<6} : exists={p.exists()}   {p}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/U

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/U

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/eylul/anaconda3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/eylul/anaconda3/lib/python3.10/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/U

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# Helpers used throughout the notebook:
def load_matrix(path):
    return pd.read_csv(path, sep=" ", index_col=0)

def get_condition(col_name):
    if 'Hypoxia' in col_name or 'Hypo' in col_name:
        return 'Hypoxia'
    elif 'Normoxia' in col_name or 'Norm' in col_name:
        return 'Normoxia'
    else:
        return 'Unknown'

def get_lane(col):
    m = re.search(r"STAR\.(\d+)_", str(col))
    return m.group(1) if m else None

def load_genes(path):
    # read only the gene names (cheap; avoids loading the full matrix)
    return set(pd.read_csv(path, sep=" ", usecols=[0], index_col=0).index)

def get_labels(df):
    return np.array([get_condition(col) for col in df.columns])

def sparsity(df):
    return float((df.values == 0).mean())

## 1. Exploratory Data Analysis

The data are single-cell RNA sequencing count matrices: genes in the rows, individual cells in the columns, each cell labelled hypoxic or normoxic. They span two breast cancer cell lines, MCF7 and HCC1806, each sequenced with two technologies, SmartSeq and DropSeq. Only SmartSeq comes in raw, unfiltered form, so we begin there, where the counts still show the data's problems before any cleaning.

### 1.1. Raw SmartSeq Analysis

First we load the metadata files to inspect.

In [ ]:
mcf7_meta = pd.read_csv(SMART / "MCF7_SmartS_MetaData.tsv", sep="\t")
hcc_meta  = pd.read_csv(SMART / "HCC1806_SmartS_MetaData.tsv", sep="\t")

print("MCF7 meta:", mcf7_meta.shape, "| cols:", list(mcf7_meta.columns))
print("HCC  meta:", hcc_meta.shape, "| cols:", list(hcc_meta.columns))
for c in ["Condition", "Hours"]:
    print("MCF7", c, dict(mcf7_meta[c].value_counts()))
    print("HCC ", c, dict(hcc_meta[c].value_counts()))
mcf7_meta.head(5)

In [ ]:
hcc_meta.head(5)

The metadata confirm the design: 383 MCF7 cells and 243 HCC1806 cells, each split about evenly between hypoxia and normoxia. Two things differ between the lines. The batch label is recorded differently (Lane for MCF7, PCR Plate for HCC1806), and the cells were kept under condition for different lengths of time, 72 hours for MCF7 against 24 for HCC1806. Because the hypoxia response builds up the longer a cell is starved of oxygen, any difference we later see between the two lines could partly reflect this exposure gap rather than the cell line itself. 

# **check if any biological explaination for this choice**

We continue by loading the raw count matrices.

In [ ]:
mcf7_raw = load_matrix(SMART / "MCF7_SmartS_Unfiltered_Data.txt")
hcc_raw  = load_matrix(SMART / "HCC1806_SmartS_Unfiltered_Data.txt")

print("MCF7 raw:   ", mcf7_raw.shape, "(genes x cells)")
print("HCC1806 raw:", hcc_raw.shape, "(genes x cells)")
print("example column:", mcf7_raw.columns[0])
print("  condition:", get_condition(mcf7_raw.columns[0]), "| lane:", get_lane(mcf7_raw.columns[0]))
mcf7_raw.iloc[:4, :3]

In [ ]:
hcc_raw.iloc[:4, :3]

In both matrices the rows are genes and the columns are individual cells, and each value is the number of reads counted for that gene in that cell. MCF7 has 22,934 genes across 383 cells, HCC1806 23,396 genes across 243 cells. The column name encodes each cell's condition and batch, so the hypoxia label is built into the data itself. Even in the first few rows most values are zero, which is the first thing worth looking at next.

In [ ]:
for name, d in [("MCF7", mcf7_raw), ("HCC1806", hcc_raw)]:
    labs = get_labels(d)
    n_h = (labs == "Hypoxia").sum()
    n_n = (labs == "Normoxia").sum()
    nan = int(d.isna().sum().sum())
    print(f"{name}: Hypoxia={n_h}, Normoxia={n_n} | NaNs={nan}")

Both datasets are close to balanced: MCF7 has 191 hypoxia and 192 normoxia cells, HCC1806 has 126 and 117, so the classes will not bias a model on their own. Neither matrix has any missing values, which means every zero is a real "not detected" count, not a gap. So the matrix is not incomplete, it is sparse, and those zeros are part of the data we have to deal with next.

In [ ]:
for name, d in [("MCF7", mcf7_raw), ("HCC1806", hcc_raw)]:
    print(f"{name}: overall zeros = {(d.values == 0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, d) in zip(axes, [("MCF7", mcf7_raw), ("HCC1806", hcc_raw)]):
    (d == 0).mean(axis=0).hist(bins=40, ax=ax)
    ax.set_title(f"{name}: zero-fraction per cell")
    ax.set_xlabel("fraction of genes = 0 in a cell")
    ax.set_ylabel("cells")
plt.tight_layout(); plt.show()

About 60% of the MCF7 matrix and 56% of HCC1806 is zero, so even before any cleaning most gene-cell entries are empty. Most cells sit between roughly 0.5 and 0.65 zeros, which is normal for this kind of data. The thing to notice is the small spike near 1.0: a handful of cells where almost every gene reads zero. Those are failed or empty wells with almost nothing captured, and they are exactly the low-quality cells that preprocessing removes. The rest is sparse but informative, which is why the non-zero values, not the zeros, carry the signal.

Next we look at how the non-zero counts are distributed and at how a gene's dropout rate depends on its average expression, which together show what kind of data these counts are.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, d) in enumerate([("MCF7", mcf7_raw), ("HCC1806", hcc_raw)]):
    nz = d.values[d.values > 0]
    gmean = d.mean(axis=1)
    dropout = (d == 0).mean(axis=1)

    axes[row, 0].hist(np.log1p(nz), bins=60)
    axes[row, 0].set_title(f"{name}: log1p(non-zero counts)")
    axes[row, 0].set_xlabel("log1p(count)")

    axes[row, 1].scatter(np.log1p(gmean), dropout, s=3, alpha=0.2)
    axes[row, 1].set_title(f"{name}: dropout curve")
    axes[row, 1].set_xlabel("log1p(mean expression)")
    axes[row, 1].set_ylabel("fraction of cells = 0")
plt.tight_layout(); plt.show()

Both cell lines show the same pattern. The non-zero counts are heavily right-skewed even on a log scale: a single read is the most common value and most detected genes sit at low counts, with only a small tail of highly expressed genes. The dropout curves show the zeros are not random. A gene's chance of being zero falls steadily as its average expression rises, so weakly expressed genes are missing in almost every cell while strongly expressed ones are detected everywhere. Because MCF7 and HCC1806 behave the same way, this reflects the data type rather than the cell line. It prompts for log-transforming and filtering out low-expression genes later.

In [ ]:
for name, d in [("MCF7", mcf7_raw), ("HCC1806", hcc_raw)]:
    dups = d[d.duplicated(keep=False)]              # genes identical to at least one other gene
    det  = (dups > 0).sum(axis=1)                   # cells each duplicate is detected in
    print(f"{name}: {dups.shape[0]} duplicate genes | "
          f"all-zero among them: {int((dups.sum(axis=1) == 0).sum())} | "
          f"detected in median {int(det.median())} cells "
          f"(all genes median: {int((d > 0).sum(axis=1).median())})")

# show one example group of identical genes (MCF7)
dups = mcf7_raw[mcf7_raw.duplicated(keep=False)]
for _, members in dups.groupby(list(dups.columns)).groups.items():
    if len(members) > 1:
        print("\nexample identical genes:", list(members),
              "| total counts:", mcf7_raw.loc[list(members)].sum(axis=1).to_dict())
        break

Checking for duplicate genes, 56 in MCF7 and 89 in HCC1806 have counts identical to another gene, and none are empty, since no gene is zero across every cell. The example report suggests duplicates are usually redundant annotations of the same gene, but that is not the case here. The flagged genes are detected in only about two cells each (median 2, against 125 for all genes), and the example pair HTR5A and RNU6-539P are unrelated genes ***quote this*** that each carry a single read in the same two cells. So these duplicates are a side effect of extreme sparsity, ultra-low genes colliding on identical count vectors by chance, rather than real annotation redundancy. They need no special deduplication, because the usual gene filter that drops genes seen in fewer than ten cells removes them anyway.

So far we have checked the shape and quality of the data, but not whether it actually carries a hypoxia signal. Before any modelling, it is worth confirming the labels mean what they should. We use genes already known to switch on under low oxygen, such as CA9, VEGFA and NDRG1: if the labels are real, these should read clearly higher in the hypoxic cells than in the normoxic ones. It also gives a first sense of how strong the biological signal is.

In [ ]:

markers = ["CA9","VEGFA","SLC2A1","LDHA","PGK1","BNIP3","NDRG1","HK2",
           "PDK1","ALDOA","P4HA1","EGLN3","ANKRD37"]

def marker_table(d):
    labs = get_labels(d)
    rows = []
    for g in markers:
        if g in d.index:
            h = d.loc[g, labs == "Hypoxia"].mean()
            n = d.loc[g, labs == "Normoxia"].mean()
            rows.append((g, round(h, 1), round(n, 1), round((h + 1) / (n + 1), 1)))
    return (pd.DataFrame(rows, columns=["gene", "Hypoxia_mean", "Normoxia_mean", "fold"])
              .sort_values("fold", ascending=False))

mcf7_mk = marker_table(mcf7_raw)
hcc_mk  = marker_table(hcc_raw)
print("MCF7:");    print(mcf7_mk.to_string(index=False))
print("\nHCC1806:"); print(hcc_mk.to_string(index=False))

Every canonical hypoxia gene reads higher under hypoxia in both lines, with folds from about 2.4x to 77x, so the labels are real and the signal is strong. Two patterns stand out. The folds are generally smaller in HCC1806 (VEGFA 2.4x against 23x in MCF7, HK2 5.4x against 63x), which fits its shorter 24-hour exposure and a less developed response. And the two lines lead with different genes: NDRG1 and HK2 are strongest in MCF7, while CA9 and EGLN3 dominate in HCC1806. So both share the core hypoxia response but emphasise different genes, an early sign that a signature learned on one line may not transfer cleanly to the other.

A risk with any sequencing experiment is that the biological condition lines up with a technical batch. If most hypoxic cells were processed in one batch and most normoxic cells in another, a model could learn the batch rather than the hypoxia signal. To rule this out, we check how hypoxia and normoxia are spread across the sequencing batches: lanes for MCF7, PCR plates for HCC1806.

In [ ]:
print("MCF7: Lane x Condition:")
print(pd.crosstab(mcf7_meta["Lane"], mcf7_meta["Condition"]))

print("\nHCC1806: PCR Plate x Condition:")
print(pd.crosstab(hcc_meta["PCR Plate"], hcc_meta["Condition"]))

MCF7 is cleanly balanced: every lane holds 48 hypoxia and 48 normoxia cells (lane 4 is 47/48), so condition is independent of lane and cannot be mistaken for the signal. HCC1806 is only roughly balanced: both conditions appear in every plate, but plate 3 leans hypoxia (39/29) and plate 4 leans normoxia (21/27). That is not a hard confound, since no plate is one-sided, but the imbalance is worth noting, as a model on HCC1806 could absorb a little plate-related variation along with the hypoxia signal.

## 2. Preprocessing of SmartSeq
**Steps:**
1. Load raw SmartSeq data
2. Remove low-quality cells (cell filtering)
3. Remove lowly-expressed genes (gene filtering)
4. Normalise library sizes
5. Select top 3000 most variable genes (HVG selection)
6. Load and validate the official train/test split files
7. Save all 8 matrices

In [ ]:
for name, df in [('MCF7', mcf7_raw), ('HCC1806', hcc_raw)]:
    labels = get_labels(df)
    n_hypo  = (labels == 'Hypoxia').sum()
    n_normo = (labels == 'Normoxia').sum()
    print(f'{name}: Hypoxia={n_hypo}, Normoxia={n_normo}')

### 2.1. Cell filtering

Some cells have very low total read counts. These are failed library preparations (empty wells or broken cells), not real biology. We need to remove them.

**Threshold: keep cells with at least 200 000 total counts.**

This threshold was chosen by looking at the scatter plot below because there is a clear gap in the library size distribution below 200 000.

In [ ]:
# QC: compute library size and genes detected per cell 
LIB_THRESH = 200000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, df) in zip(axes, [('MCF7', mcf7_raw), ('HCC1806', hcc_raw)]):
 lib_sizes = df.sum(axis=0) # total counts per cell
 genes_found = (df > 0).sum(axis=0) # genes with count > 0 per cell
 labels = get_labels(df)

 # Plot each condition
 for cond in ['Hypoxia', 'Normoxia']:
 mask = [l == cond for l in labels]
 ax.scatter(lib_sizes[mask], genes_found[mask],
 s=15, alpha=0.7, label=cond)

 ax.axvline(LIB_THRESH, color='black', ls='--', lw=1.5,
 label=f'Threshold ({LIB_THRESH:,})')
 ax.set_title(f'{name} - library size vs genes detected')
 ax.set_xlabel('Library size (total counts)')
 ax.set_ylabel('Number of genes detected')
 ax.legend()

plt.suptitle('Cell QC: before filtering')
plt.tight_layout()
plt.show()

In [ ]:
#Remove cells with library size < 200,000

def filter_cells(df, threshold, name):
    lib_sizes = df.sum(axis=0)
    keep = lib_sizes[lib_sizes >= threshold].index
    n_removed = df.shape[1] - len(keep)
    print(f'{name}: removed {n_removed} cells, kept {len(keep)} out of {df.shape[1]}')
    return df[keep]

mcf7_cf = filter_cells(mcf7_raw, LIB_THRESH, 'MCF7')
hcc_cf  = filter_cells(hcc_raw,  LIB_THRESH, 'HCC1806')

# Check class balance is still ok
print()
for name, df in [('MCF7', mcf7_cf), ('HCC1806', hcc_cf)]:
    labels = get_labels(df)
    n_hypo  = (labels == 'Hypoxia').sum()
    n_normo = (labels == 'Normoxia').sum()
    total   = n_hypo + n_normo
    print(f'{name} after cell filter: Hypoxia={n_hypo} ({n_hypo/total:.1%}),  '
          f'Normoxia={n_normo} ({n_normo/total:.1%})')

In [ ]:
#Plot library sizes before and after cell filtering
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

datasets = [('MCF7', mcf7_raw, mcf7_cf), ('HCC1806', hcc_raw, hcc_cf)]

for row, (name, raw, filtered) in enumerate(datasets):
 for col, (df, title) in enumerate([(raw, 'Before'), (filtered, 'After')]):
 ax = axes[row, col]
 labels = get_labels(df)
 libs = df.sum(axis=0)

 for cond in ['Hypoxia', 'Normoxia']:
 mask = [l == cond for l in labels]
 ax.hist(libs[mask], bins=30, alpha=0.65,
 label=cond, edgecolor='white')

 if title == 'Before':
 ax.axvline(LIB_THRESH, color='black', ls='--', lw=1.3,
 label=f'Threshold ({LIB_THRESH:,})')

 ax.set_title(f'{name} - {title} cell filter')
 ax.set_xlabel('Library size')
 ax.set_ylabel('Number of cells')
 ax.legend(fontsize=8)

plt.suptitle('Library sizes:before vs after cell filtering')
plt.tight_layout()
plt.show()

### 2.2 Gene filtering

Many genes are detected in only a handful of cells. These genes are mostly noise, they don't have enough observations to be useful for classification.

**Threshold: keep genes detected in at least 10 cells.**

In [ ]:
MIN_CELLS = 10

def filter_genes(df, min_cells, name):
    # For each gene, count how many cells have a non-zero value
    cells_per_gene = (df > 0).sum(axis=1)
    keep = cells_per_gene[cells_per_gene >= min_cells].index
    n_removed = df.shape[0] - len(keep)
    print(f'{name}: removed {n_removed} genes, kept {len(keep)} out of {df.shape[0]}')
    return df.loc[keep]

mcf7_gf = filter_genes(mcf7_cf, MIN_CELLS, 'MCF7')
hcc_gf  = filter_genes(hcc_cf,  MIN_CELLS, 'HCC1806')

# Summary table
print()
print(f'{"Dataset":<15}  {"Genes (raw)":>12}  {"Genes (kept)":>13}  {"Cells (kept)":>13}')
print('-' * 57)
print(f'{"MCF7":<15}  {mcf7_raw.shape[0]:>12,}  {mcf7_gf.shape[0]:>13,}  {mcf7_gf.shape[1]:>13,}')
print(f'{"HCC1806":<15}  {hcc_raw.shape[0]:>12,}  {hcc_gf.shape[0]:>13,}  {hcc_gf.shape[1]:>13,}')

In [ ]:
#Plot: how many cells express each gene, before and after gene filter
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, cf, gf) in zip(axes, [('MCF7', mcf7_cf, mcf7_gf),
                                       ('HCC1806', hcc_cf, hcc_gf)]):
    before = (cf > 0).sum(axis=1)   # cells per gene, before
    after  = (gf > 0).sum(axis=1)   # cells per gene, after

    ax.hist(before, bins=40, alpha=0.5,
            label='Before filter', edgecolor='white')
    ax.hist(after,  bins=40, alpha=0.8,
            label='After filter',  edgecolor='white')
    ax.axvline(MIN_CELLS, color='black', ls='--', lw=1.3,
               label=f'Threshold ({MIN_CELLS} cells)')
    ax.set_title(f'{name}')
    ax.set_xlabel('Number of cells expressing the gene')
    ax.set_ylabel('Number of genes')
    ax.legend(fontsize=9)

plt.suptitle('Gene filtering: cells per gene distribution')
plt.tight_layout()
plt.show()

### 2.3 Library size normalisation

Even after cell filtering, cells still have different total read counts. This is technical: cells that happen to be sequenced more deeply will appear to express every gene at a higher level, even if biology is identical.

**Method: scale every cell so its total equals the median size.**

$$x_{\text{norm}} = x \times \frac{\text{median library size}}{\text{cell library size}}$$

We use the **median** (not the mean) because it is more robust to a few very large outlier cells.

In [ ]:
def normalise(df, name):
    lib_sizes  = df.sum(axis=0)          # total counts per cell
    median_lib = lib_sizes.median()       # target: the median library size

    print(f'{name}: median library size = {median_lib:,.0f}')

    # Scale each cell
    scale_factors = median_lib / lib_sizes
    df_norm = df.multiply(scale_factors, axis=1).round(0).astype(int)

    return df_norm

mcf7_norm = normalise(mcf7_gf, 'MCF7')
hcc_norm  = normalise(hcc_gf,  'HCC1806')

# Verify: after normalisation all library sizes should be roughly equal
print()
print('Library size after normalisation (should all be similar):')
for name, df in [('MCF7', mcf7_norm), ('HCC1806', hcc_norm)]:
    libs = df.sum(axis=0)
    print(f'  {name}: min={libs.min():,.0f}  median={libs.median():,.0f}  max={libs.max():,.0f}')

In [ ]:
#Plot: library sizes before and after normalisation
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

datasets = [('MCF7', mcf7_gf, mcf7_norm), ('HCC1806', hcc_gf, hcc_norm)]

for row, (name, before_df, after_df) in enumerate(datasets):
 labels = get_labels(before_df)

 for col, (df, title) in enumerate([(before_df, 'Before'), (after_df, 'After')]):
 ax = axes[row, col]
 libs = df.sum(axis=0)

 for cond in ['Hypoxia', 'Normoxia']:
 mask = [l == cond for l in labels]
 ax.hist(libs[mask], bins=30, alpha=0.65,
 label=cond, edgecolor='white')

 ax.set_title(f'{name} - {title} normalisation')
 ax.set_xlabel('Total counts per cell')
 ax.set_ylabel('Number of cells')
 ax.legend(fontsize=8)

plt.suptitle('Library sizes: before vs after normalisation')
plt.tight_layout()
plt.show()

### 2.4 Top 3000 highly variable gene selection

We still have around 17 000 genes after filtering. Most of them have similar expression in Hypoxia and Normoxia and carry no useful signal. We keep only the **3000 genes with the highest variance**: those that differ the most across cells and so are most likely to help distinguish the two conditions.

We choose 3000 to match the DropSeq matrices.

In [ ]:
N_HVG = 3000

# Compute variance of each gene across all cells, pick the top 3000
mcf7_var       = mcf7_norm.var(axis=1)
mcf7_top_genes = mcf7_var.nlargest(N_HVG).index

hcc_var        = hcc_norm.var(axis=1)
hcc_top_genes  = hcc_var.nlargest(N_HVG).index

print('Top 3000 HVGs selected.')
print(f'MCF7 top gene: {mcf7_top_genes[0]}  (variance = {mcf7_var[mcf7_top_genes[0]]:.1f})')
print(f'HCC1806 top gene: {hcc_top_genes[0]}  (variance = {hcc_var[hcc_top_genes[0]]:.1f})')

In [ ]:
#Mean-variance plot: show which genes were selected
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, df, top_genes) in zip(axes, [
 ('MCF7', mcf7_norm, mcf7_top_genes),
 ('HCC1806', hcc_norm, hcc_top_genes),
]):
 gene_var = df.var(axis=1)
 gene_mean = df.mean(axis=1)

 # All genes in blue
 not_selected = [g for g in gene_var.index if g not in top_genes]
 ax.scatter(gene_mean[not_selected], gene_var[not_selected],
 s=3, alpha=0.2, color='lightblue', label='Not selected')

 # Top 3000 in red
 ax.scatter(gene_mean[top_genes], gene_var[top_genes],
 s=4, alpha=0.5, color='lightcoral', label=f'Top {N_HVG} HVGs')
 
 ax.set_xscale('log')
 ax.set_yscale('log')
 ax.set_xlabel('Mean expression')
 ax.set_ylabel('Variance')
 ax.set_title(f'{name} - mean vs variance')
 ax.legend(markerscale=3, fontsize=9)

plt.suptitle(f'HVG selection - top {N_HVG} genes highlighted', fontweight='bold')
plt.tight_layout()
plt.show()

#Top 5 most variable genes in MCF7
print("\nTop 5 most variable genes in MCF7:")
print(mcf7_var.nlargest(5).index.tolist())

# Top 5 most variable genes in HCC1806
print("\nTop 5 most variable genes in HCC1806:")
print(hcc_var.nlargest(5).index.tolist())

### 2.5 Load official train/test split files

The course provides the final train and test files already split. We load them here and check:
- Do train and test share the same 3000 genes?
- Is the class balance reasonable?

In [ ]:
# Load SmartSeq train/test files
mcf7_ss_train  = load_matrix(SMART / 'MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt')
mcf7_ss_test   = load_matrix(SMART / 'MCF7_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt')
hcc_ss_train   = load_matrix(SMART / 'HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt')
hcc_ss_test    = load_matrix(SMART / 'HCC1806_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt')

# Load DropSeq train/test files
mcf7_ds_train  = load_matrix(DROP / 'MCF7_Filtered_Normalised_3000_Data_train.txt')
mcf7_ds_test   = load_matrix(DROP / 'MCF7_Filtered_Normalised_3000_Data_test_anonim.txt')
hcc_ds_train   = load_matrix(DROP / 'HCC1806_Filtered_Normalised_3000_Data_train.txt')
hcc_ds_test    = load_matrix(DROP / 'HCC1806_Filtered_Normalised_3000_Data_test_anonim.txt')

print('All 8 matrices loaded.')
print()
print(f'{"Dataset":<25}  {"Train cells":>12}  {"Test cells":>11}  {"Genes":>7}')
print('-' * 60)
for name, train, test in [
    ('SmartSeq MCF7',    mcf7_ss_train, mcf7_ss_test),
    ('SmartSeq HCC1806', hcc_ss_train,  hcc_ss_test),
    ('DropSeq MCF7',     mcf7_ds_train, mcf7_ds_test),
    ('DropSeq HCC1806',  hcc_ds_train,  hcc_ds_test),
]:
    print(f'{name:<25}  {train.shape[1]:>12,}  {test.shape[1]:>11,}  {train.shape[0]:>7,}')

In [ ]:
# Check gene sets match between train and test
print('Gene set consistency (train vs test):')
for name, train, test in [
    ('SmartSeq MCF7',    mcf7_ss_train, mcf7_ss_test),
    ('SmartSeq HCC1806', hcc_ss_train,  hcc_ss_test),
    ('DropSeq MCF7',     mcf7_ds_train, mcf7_ds_test),
    ('DropSeq HCC1806',  hcc_ds_train,  hcc_ds_test),
]:
    same = set(train.index) == set(test.index)
    print(f'  {name}: genes match = {same}')

print()
print('Class balance in training sets:')
for name, train in [
    ('SmartSeq MCF7',    mcf7_ss_train),
    ('SmartSeq HCC1806', hcc_ss_train),
    ('DropSeq MCF7',     mcf7_ds_train),
    ('DropSeq HCC1806',  hcc_ds_train),
]:
    labels  = get_labels(train)
    n_hypo  = (labels == 'Hypoxia').sum()
    n_normo = (labels == 'Normoxia').sum()
    total   = n_hypo + n_normo
    print(f'  {name}: Hypoxia={n_hypo} ({n_hypo/total:.1%}),  Normoxia={n_normo} ({n_normo/total:.1%})')

In [ ]:
#Class balance bar chart
names  = ['SmartSeq\nMCF7', 'SmartSeq\nHCC1806', 'DropSeq\nMCF7', 'DropSeq\nHCC1806']
trains = [mcf7_ss_train, hcc_ss_train, mcf7_ds_train, hcc_ds_train]

hypo_pcts  = []
normo_pcts = []
for train in trains:
    labels  = get_labels(train)
    n_hypo  = (labels == 'Hypoxia').sum()
    n_normo = (labels == 'Normoxia').sum()
    total   = n_hypo + n_normo
    hypo_pcts.append(100 * n_hypo  / total)
    normo_pcts.append(100 * n_normo / total)

x   = range(len(names))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, hypo_pcts,  label='Hypoxia',  color='lightcoral')
ax.bar(x, normo_pcts, bottom=hypo_pcts, label='Normoxia', color='lightblue')
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('% of training cells')
ax.set_ylim(0, 100)
ax.set_title('Class balance in all four training sets', fontweight='bold')
ax.legend()

for i, (h, n) in enumerate(zip(hypo_pcts, normo_pcts)):
    ax.text(i, h/2,   f'{h:.1f}%', ha='center', va='center', color='white', fontweight='bold')
    ax.text(i, h+n/2, f'{n:.1f}%', ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

### 2.6 Preprocessing summary

| Step | What we did | Why |
|---|---|---|
| Cell filtering | Removed cells with < 200,000 total counts | Failed library preps / empty wells |
| Gene filtering | Removed genes detected in < 10 cells | Too few observations to be useful |
| Normalisation | Scaled all cells to the same total count (median) | Removes sequencing depth differences |
| HVG selection | Kept the 3,000 most variable genes | Reduces noise; matches DropSeq |

### Data flow

| | MCF7 genes | MCF7 cells | HCC1806 genes | HCC1806 cells |
|---|---|---|---|---|
| Raw | 22,934 | 383 | 23,396 | 243 |
| After cell filter | 22,934 | 326 | 23,396 | 234 |
| After gene filter | 17,666 | 326 | 17,726 | 234 |
| After HVG selection | 3,000 | 326 | 3,000 | 234 |
| Train set (final) | 3,000 | 250 | 3,000 | 182 |
| Test set (final) | 3,000 | 63 | 3,000 | 45 |

## 3. Comparison: our preprocessing vs the provided filtered files

In section 2 we replicated the preprocessing pipeline from scratch on the raw SmartSeq data. Here we compare our output against the curator-provided filtered and normalised files to validate our steps. Differences can arise from slightly different thresholds, HVG selection methods, or normalisation targets: identifying them tells us how close our pipeline is to the reference and whether the provided files should be preferred as the canonical input for downstream analysis.

In [ ]:
for name, ours_norm, our_top_genes, ref in [
    ('MCF7',    mcf7_norm, mcf7_top_genes, mcf7_ss_train),
    ('HCC1806', hcc_norm,  hcc_top_genes,  hcc_ss_train),
]:
    ours = ours_norm.loc[our_top_genes]  # genes x cells, our preprocessing

    our_genes = set(ours.index)
    ref_genes = set(ref.index)
    shared    = our_genes & ref_genes

    print(f"{name}")
    print(f"  Our shape:      {ours.shape}  |  Ref shape: {ref.shape}")
    print(f"  Shared genes:   {len(shared)}")
    print(f"  Only in ours:   {len(our_genes - ref_genes)}")
    print(f"  Only in ref:    {len(ref_genes - our_genes)}")

    if shared:
        # align on shared genes and shared cells
        shared_cells = list(set(ours.columns) & set(ref.columns))
        shared_genes = list(shared)
        if shared_cells:
            diff = (ours.loc[shared_genes, shared_cells] - ref.loc[shared_genes, shared_cells]).abs()
            print(f"  Shared cells: {len(shared_cells)}")
            print(f"  Max value diff: {diff.values.max():.6f}")
            print(f"  Mean value diff: {diff.values.mean():.6f}")
        else:
            print(f"  No shared cell barcodes (different splits or naming)")
    print()

Despite the similarity in preprocessing steps, we proceed with the curator-provided filtered files for all downstream analysis to ensure consistency with the course challenge and comparability across groups.

## 4. Filtered data analysis
We initially diagnosed the raw data, but only SmartSeq exists in raw form. From here we switch to the processed files provided with the course, which cover all four datasets, so the two cell lines and the two technologies can finally be compared on equal footing. Everything is now at the same processing stage, so the differences we see are real, not artefacts of one set being cleaned and another not.

In [ ]:
processed = [
    ("SmartSeq MCF7",    SMART / "MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt"),
    ("SmartSeq HCC1806", SMART / "HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt"),
    ("DropSeq MCF7",     DROP / "MCF7_Filtered_Normalised_3000_Data_train.txt"),
    ("DropSeq HCC1806",  DROP / "HCC1806_Filtered_Normalised_3000_Data_train.txt"),
]

for name, path in processed:
    d = load_matrix(path)
    print(f"=== {name} === {d.shape} (genes x cells)")
    display(d.iloc[:4, :3])
    del d



A first look at the four filtered matrices shows they share a layout, 3000 genes by a varying number of cells, but little else. The cell labels differ by technology: SmartSeq keeps the full sequencing filenames, which still carry the batch information (lane for MCF7, plate for HCC1806), while DropSeq uses short barcodes with no batch information at all. The values differ just as much, with SmartSeq cells holding large graded counts and DropSeq cells nearly empty, their few non-zero entries being single reads on the most abundant genes such as MALAT1

In [ ]:
rows = []
for name, path in processed:
    d = load_matrix(path)
    labs = get_labels(d)
    rows.append({
        "dataset": name,
        "genes": d.shape[0],
        "cells": d.shape[1],
        "%zeros": round((d.values == 0).mean() * 100, 1),
        "median_lib": round(float(d.sum(axis=0).median()), 1),
        "Hypoxia": int((labs == "Hypoxia").sum()),
        "Normoxia": int((labs == "Normoxia").sum()),
    })
    del d

glance = pd.DataFrame(rows)
glance


The gene panel is fixed at 3,000 for all four, but otherwise they fall into two regimes. SmartSeq has few cells (250 and 182) sequenced deeply, with median library sizes of 330k - 500k and moderate sparsity (64 - 71%). DropSeq has tens of thousands of cells (21 626 and 14 682) sequenced very shallowly, with median library sizes near 100 and 97.5% sparsity, so depth per cell differs by several thousand-fold. Class balance also splits by technology: SmartSeq is close to even, while DropSeq is skewed in opposite directions, MCF7 leaning normoxia (8921 vs 12 705) and HCC1806 leaning hypoxia (8899 vs 5783). That skew is worth flagging for the modelling step.

To see the sparsity as a pattern rather than a single percentage, we plot a 100-gene by 100-cell corner of each matrix, with black marking a detected gene and white a zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, path) in zip(axes, [processed[0], processed[2]]):   # SmartSeq MCF7 vs DropSeq MCF7
    d = load_matrix(path)
    ax.imshow(d.iloc[:100, :100] > 0, aspect="auto", cmap="Greys", interpolation="nearest")
    ax.set_title(f"{name}: 100x100 slice (black = detected)")
    ax.set_xlabel("cells"); ax.set_ylabel("genes")
    del d
plt.tight_layout(); plt.show()

The sparsity plots showed how often genes are detected; this looks at how large those detected values are. For each dataset we take every non-zero value and plot its distribution on a log scale, so the magnitude of expression can be compared across the two technologies at the same processing stage.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
rng = np.random.default_rng(0)
for ax, (name, path) in zip(axes.flat, processed):
    d = load_matrix(path)
    vals = d.values[d.values > 0]
    if vals.size > 500000:                 # sample to keep the big DropSeq matrices fast
        vals = rng.choice(vals, 500000, replace=False)
    ax.hist(np.log1p(vals), bins=50, color="#4C72B0")
    ax.set_title(name)
    ax.set_xlabel("log1p(non-zero value)"); ax.set_ylabel("count")
    del d
plt.tight_layout(); plt.show()

The two technologies behave very differently. In SmartSeq the detected values span a wide range, from single reads up to high counts, with log values reaching about 10, giving a broad graded distribution in both lines. In DropSeq almost every detected value is a single read: the distribution collapses to a spike at the low end, with a maximum log value of only about 3 to 5. So DropSeq detections are not only rare but tiny, a gene is usually either absent or seen exactly once, leaving the data close to binary, while SmartSeq records graded expression. A DropSeq cell therefore carries far less quantitative information per gene, which limits what a model can learn from it and helps explain why DropSeq is the harder case.

### Shared gene panel

Each dataset keeps its own 3,000 most informative genes, chosen from that dataset alone. Nothing forces those choices to agree, so the four panels may not even describe the same genes. This matters directly for the project's central question: if a model is trained on one dataset and tested on another, it can only use genes the two have in common, so the overlap between panels sets a hard ceiling on how well any model can transfer. Here we measure that overlap, pairwise and across all four.

In [ ]:
panels = {}
for name, path in processed:
    panels[name] = set(pd.read_csv(path, sep=" ", usecols=[0], index_col=0).index)

names = list(panels)
overlap = pd.DataFrame(index=names, columns=names, dtype=int)
for a in names:
    for b in names:
        overlap.loc[a, b] = len(panels[a] & panels[b])

print("Shared genes between panels (out of 3000):")
print(overlap)

core = set.intersection(*panels.values())
print(f"\nGenes shared by all four datasets: {len(core)}")

The panels overlap remarkably little. Out of 3000 genes, the two SmartSeq datasets share 1208 and the two DropSeq datasets 834, so even within one technology the two cell lines agree on at most 40% of their genes. Across technologies it is worse: the same cell line shares only about 500 genes between SmartSeq and DropSeq (496 for MCF7, 516 for HCC1806), roughly 17%. Only 70 genes, 2.3%, are common to all four. The clearest pattern is that technology divides the panels more than cell line does, since same-technology pairs share more genes than the same cell line measured across technologies. Which 3,000 genes look most informative is driven more by the sequencing platform than by the biology. This sets a hard ceiling on generalization: a model trained on one dataset and applied to another can only use the genes they share, so a cross-technology model has only about 17% of its features even present in the target. It also predicts the asymmetry the generalization section reports, transfer across cell lines, with more shared genes, should beat transfer across technologies, with fewer.

The obvious question is whether those few shared genes are the biologically important ones. If the canonical hypoxia markers survive into every panel, then despite the low overlap there is still a stable biological core a cross-dataset model could lean on. If they do not, even the key hypoxia genes are being dropped inconsistently, which would make generalisation harder still. Here we check which markers are retained in each panel and which reach the shared core.

In [ ]:
markers = ["CA9","VEGFA","SLC2A1","LDHA","PGK1","BNIP3","NDRG1","HK2",
           "PDK1","ALDOA","P4HA1","EGLN3","ANKRD37"]

rows = []
for g in markers:
    row = {"gene": g}
    for name in panels:
        row[name] = "yes" if g in panels[name] else "-"
    row["all 4"] = "yes" if g in core else "-"
    rows.append(row)

marker_panel = pd.DataFrame(rows).set_index("gene")
print("Hypoxia markers retained in each panel:")
print(marker_panel)
print(f"\nMarkers in the 70-gene shared core: {sorted(set(markers) & core)}")

The result is split by technology. The two SmartSeq panels keep almost every canonical marker (12 of 13 for MCF7, 11 for HCC1806), so on deep data the variance-based selection naturally retains the hypoxia biology. DropSeq is inconsistent: HCC1806 still keeps 9 of the 13, but MCF7 keeps only one, PGK1. So under heavy dropout the gene-selection step can strip out almost all the textbook hypoxia genes, and it does so differently from one dataset to the next. Only PGK1, a broadly expressed glycolytic gene, survives into all four panels, so the entire shared core holds just a single canonical marker. Two points follow. The biology is not gone, the earlier fold changes show the markers respond, but variance selection on sparse data fails to rank the more hypoxia-specific, lowly detected genes, so they drop out. And the shared core is biologically thin: a model limited to genes common to all four would have almost none of the known hypoxia signature, which compounds the overlap problem from 3.5.

### Gene to gene correlation

So far we have treated the 3000 genes as 3000 separate features. But genes rarely act alone: those in the same pathway tend to rise and fall together, so many of the features may be carrying the same information. Before any modelling it is worth asking how independent the genes really are, because if they are highly correlated, the data holds far fewer effective dimensions than 3000, and dimensionality reduction becomes the natural next step. We measure this with gene-to-gene correlation, comparing a deep dataset (SmartSeq) against a shallow one (DropSeq).

In [ ]:
results = {}
for name, path in [("SmartSeq MCF7", processed[0][1]), ("DropSeq MCF7", processed[2][1])]:
    d = load_matrix(path)
    x = np.log1p(d.values)
    idx = np.argsort(x.var(axis=1))[-500:]        # top 500 most variable genes
    c = np.corrcoef(x[idx])
    off = c[np.triu_indices_from(c, k=1)]
    off = off[~np.isnan(off)]
    results[name] = off
    print(f"{name}: median |r| = {np.median(np.abs(off)):.3f}, "
          f"fraction |r|>0.3 = {(np.abs(off) > 0.3).mean():.1%}")
    del d, x, c

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, results):
    ax.hist(results[name], bins=60, color="#4C72B0")
    ax.set_title(f"{name}: gene-gene correlations")
    ax.set_xlabel("pairwise r (top 500 variable genes)"); ax.set_xlim(-1, 1)
plt.tight_layout(); plt.show()

The two technologies look completely different. In SmartSeq the gene-gene correlations spread widely: median absolute correlation 0.21, with 31% of gene pairs above 0.3, so much of the 3000-gene panel moves together in co-expression modules. The features are far from independent, so the data carries many fewer effective dimensions than 3000 and dimensionality reduction is well justified. In DropSeq the same measurement collapses to a spike at zero (median 0.019, almost no pairs above 0.3), not because the biology is gone but because dropout removes the co-occurrence needed to see it: with so few genes detected per cell, pairs rarely appear together and their correlations wash out. So reducing dimensions should work cleanly on SmartSeq, where there is real structure to compress, but has little to grip onto in DropSeq, which already predicts the unsupervised step will behave very differently across the two technologies.

## 5. Unsupervised Learning

The EDA shows us that the hypoxia signal is real and strong, but that the two technologies produce very different data: SmartSeq has rich, detailed, and heavily expressed data with high gene-gene correlation, while DropSeq is dominated by dropout and almost binary values, making it sparse and noisy. Unsupervised learning asks the following: **Can we recover the Hypoxia/Normoxia structure from expression patterns alone, without ever looking at labels?**

### 5.1 Load Data

We load the data again directly from the curator-provided files to ensure consistency across all groups. We work only with the train splits for unsupervised learning to ensure no information is leaked and that there is no bias in the analysis.

In [ ]:
#load matrices
datasets = {
    ('SmartSeq', 'MCF7'): load_matrix(SMART/'MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt'),
    ('SmartSeq', 'HCC1806'): load_matrix(SMART/'HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt'),
    ('DropSeq', 'MCF7'): load_matrix(DROP/'MCF7_Filtered_Normalised_3000_Data_train.txt'),
    ('DropSeq', 'HCC1806'): load_matrix(DROP/'HCC1806_Filtered_Normalised_3000_Data_train.txt'),
}

print(f"{'Technology':<12} {'Cell line':<10} {'Genes':>7} {'Cells':>8}  dtype")
print('─' * 55)
for (tech, line), df in datasets.items():
    print(f"{tech:<12} {line:<10} {df.shape[0]:>7,} {df.shape[1]:>8,}  {df.dtypes.iloc[0]}")

### 5.2 Preprocessing for unsupervised analysis

We do two steps:
1. **Log transform** (both SmartSeq and DropSeq): `log2(x + 1)` compresses the dynamic range and stabilises variance, to make PCA and distance metrics more meaningful.
2. **Z-score per gene** across all cells: brings every gene to mean=0, std=1 so that highly-expressed genes don't dominate PCA axes purely because of their scale.

In [ ]:
def preprocess(df: pd.DataFrame) -> np.ndarray:
    X = df.values.T.astype(float) #transpose
    X = np.log2(X + 1)
    X_scaled = StandardScaler().fit_transform(X) #z-score per gene
    return X_scaled

X_dict = {}
cond_dict = {}

for (tech, line), df in datasets.items():
    X_dict[(tech, line)] = preprocess(df)
    cond_dict[(tech, line)] = get_labels(df)

print('Preprocessing complete.')
for key, X in X_dict.items():
    print(f"{key[0]} {key[1]}:{X.shape}(cells × genes)")

### 5.3 Cell-cell Pearson correlation heatmaps

These correlation heatmaps check whether cells of the same condition are more similar to each other than to cells of the other condition. If the biology is real, hypoxic cells will correlate with each other and normoxic with each other: each entry (i,j) is the Pearson correlation between cell i and cell j across all 3000 genes. 

*Note:* Since DropSeq has too many cells (15000-21000) to plot the full matrix, we subsample 400 cells, 200 per condition, for the visualisation, but other analyses use the full dataset.

In [ ]:
MAX_HEATMAP = 400 #cells to display in heatmap

COND_COL = {'Hypoxia': '#e05c5c', 'Normoxia': '#5c7de0'}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, ((tech, line), X) in zip(axes, X_dict.items()):
 conds = cond_dict[(tech, line)]

 if X.shape[0] > MAX_HEATMAP:
 idx = RNG.choice(X.shape[0], MAX_HEATMAP, replace=False)
 X_s, c_s = X[idx], conds[idx]
 note = f'n={len(idx)} sampled'
 else:
 X_s, c_s = X, conds
 note = f'n={X.shape[0]}'

 order = np.argsort(c_s, kind='stable')
 X_s, c_s = X_s[order], c_s[order]

 corr = np.corrcoef(X_s)

 im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
 plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Pearson r')

 ax.set_title(f'{tech} {line} ({note})', fontweight='bold')
 ax.set_xlabel('Cell (sorted by condition)')
 ax.set_ylabel('Cell (sorted by condition)')

 legend_elems = [mpatches.Patch(facecolor=COND_COL['Hypoxia'], label='Hypoxia'), mpatches.Patch(facecolor=COND_COL['Normoxia'], label='Normoxia')]
 ax.legend(handles=legend_elems, loc='lower right', fontsize=8)

fig.suptitle('Cell - cell Pearson correlation heatmaps (sorted by condition)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In the heatmap, the red entries represent similar expression, the blue different expression, and the white represents no relationship. The cells are sorted Hypoxia first then Normoxia. 

**SmartSeq MCF7:** Shows a clear correlation between hypoxia with hypoxia (red top-left block) and normoxia with normoxia (red bottom-right block), and clear dissimilarity between the two (blue bottom-left and top-right blocks).

**SmartSeq HCC1806:** Sas the same pattern as SmartSeq MCF7 but weaker, which could be due to HCC being kept under conditions for less than MCF.

**DropSeq:** Both have nearly no visible structure (almost completely white). This could be because of the dropout, which collapses correlations to 0 regardless of condition.

Overall, the block structure confirms that hypoxia dominates the cell-cell expression differences. The bright red dots off the diagonal in SmartSeq HCC1806 suggests a small number of cells might behave differently from the rest, i.e. they might be outliers, which we investigate next.

#### 5.3.1 Flag outlier cells

A cell is flagged as an outlier if its mean Pearson correlation with all other cells is more than 3 standard deviations below the dataset mean, so a cell that is unusually dissimilar to everyone else. We do this because they can distort PCA results by pulling entire clusters towards them. It also helps to decide whether to remove them before training in the classifier section.

In [ ]:
SD_THRESHOLD = 3.0
outlier_rows = []

for (tech, line), X in X_dict.items():
    corr = np.corrcoef(X)
    np.fill_diagonal(corr, np.nan) #exclude self-correlation
    mean_corr = np.nanmean(corr, axis=1) #mean correlation of each cell vs all others

    mu, sd = np.nanmean(mean_corr), np.nanstd(mean_corr)
    threshold = mu - SD_THRESHOLD * sd
    outlier_mask = mean_corr < threshold

    col_names = datasets[(tech, line)].columns
    n_out = outlier_mask.sum()
    print(f"{tech:10s} {line:8s}: μ={mu:.3f}, σ={sd:.3f}, threshold={threshold:.3f} → {n_out} outlier(s)")

    for idx in np.where(outlier_mask)[0]:
        outlier_rows.append({'Technology': tech, 'Cell line': line,
                              'Cell': col_names[idx],
                              'Mean corr': round(float(mean_corr[idx]), 4),
                              'Threshold': round(float(threshold), 4)})

outlier_df = pd.DataFrame(outlier_rows)
print()
if outlier_df.empty:
    print('No outliers detected across all datasets.')
else:
    print(outlier_df.to_string(index=False))

All datasets are clean except for SmartSeq HCC1806, where all are in Hypoxia. They could be failed cell preparations, cells that did not respond as strongly to hypoxia, or contamination, i.e. technical failures. DropSeq having no outliers makes sense, since the correlations are near 0 for everyone due to dropout.

These are flagged but not removed, so we can keep an eye out for distortion in PCA or in clustering results.

### 5.4 Principal Component Analysis (PCA)

PCA rotates the 3000-dimensional gene expression space to find the directions of maximum variance. Because we z-scored the datasets first, every gene contributes equally so PCA does not capture scale differences instead of biology. We have two outputs per dataset:

**Scree Plot:** Graphs the variance explained per PC and shows a cumulative curve, so we can see how many PCs we need to explain the main structure.
**PC1 vs PC2 scatter:** Tests if PCA separates the two conditions without ever seeing labels.

In [ ]:
N_COMPONENTS = 30 #compute top-30 PCs

pca_dict = {} #fitted PCA objects
scores_dict = {} #PCA-transformed coordinates(cells × PCs)

for (tech, line), X in X_dict.items():
    n_comp = min(N_COMPONENTS, X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    scores = pca.fit_transform(X)
    pca_dict[(tech, line)] = pca
    scores_dict[(tech, line)] = scores
    total_var = pca.explained_variance_ratio_.sum() * 100

    print(f"{tech:10s} {line:8s}: top-{n_comp} PCs explain {total_var:.1f}% of variance")

*Note:* The gap between explained variance for SmartSeq and DropSeq is expected since DropSeq's high dropout spreads variance across thousands of dimensions rather than concentrating it in a few meaningful components.

#### 5.4.1 Scree plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, ((tech, line), pca) in zip(axes, pca_dict.items()):
 n_show = min(20, len(pca.explained_variance_ratio_))
 ev = pca.explained_variance_ratio_[:n_show] * 100
 cumev = np.cumsum(pca.explained_variance_ratio_) * 100

 ax2 = ax.twinx()
 ax.bar(range(1, n_show + 1), ev, color='#4a90d9', alpha=0.75, label='Per-PC')
 ax2.plot(range(1, n_show + 1), cumev[:n_show], color='#e07b4a', marker='o', ms=4, lw=1.8, label='Cumulative')

 ax.set_xlabel('Principal Component')
 ax.set_ylabel('% Variance Explained', color='#4a90d9')
 ax2.set_ylabel('Cumulative % Variance', color='#e07b4a')
 ax.set_title(f'{tech} {line}', fontweight='bold')
 ax.set_xticks(range(1, n_show + 1, 2))

 h1, l1 = ax.get_legend_handles_labels()
 h2, l2 = ax2.get_legend_handles_labels()
 ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper right')

fig.suptitle('Scree plots - variance explained per PC', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

The blue bars show the explained variance per PC (numbers are on the y-axis), and the orange line shows the cumulative total variance. A steep drop after the first few bars (elbow) means the hypoxia response is concentrated in just a few dimensions rather than spread across all 3000. 

**SmartSeq MCF7:** PC1 alone explains over 10%, but there's a clear elbow after PC2. Meaning the hypoxia response is the main explanation for this variance. 

**SmartSeq HCC1806**: There's an elbow after PC2 but it's softer. PC1 alone only explains about 4.5%, so the hypoxia signal is not as strong an explanation.

**DropSeq:** No real elbows, cumulative variance rises almost linearly, and bars are all quite small. There's only around 3% variance explained cumulatively at top 30 PCs, which is due to dropout which spreads variance across thousands of dimensions.

Since no dataset reaches 80% variance even with 30 PCs, we will use less for clustering.

### 5.4.2 PC1 vs PC2 coloured by Hypoxia / Normoxia

In [ ]:
MAX_SCATTER = 500

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for ax, ((tech, line), scores) in zip(axes, scores_dict.items()):
    pca = pca_dict[(tech, line)]
    conds = cond_dict[(tech, line)]

    if scores.shape[0] > MAX_SCATTER:
        idx = RNG.choice(scores.shape[0], MAX_SCATTER, replace=False)
        s_plot = scores[idx, :2]
        c_plot = conds[idx]
        note = f'n={len(idx)} sampled'
    else:
        s_plot, c_plot = scores[:, :2], conds
        note = f'n={scores.shape[0]}'

    for cond, color in COND_COL.items():
        mask = c_plot == cond
        ax.scatter(s_plot[mask, 0], s_plot[mask, 1], color=color, alpha=0.55, s=18, label=cond, rasterized=True)

    ev1 = pca.explained_variance_ratio_[0] * 100
    ev2 = pca.explained_variance_ratio_[1] * 100
    ax.set_xlabel(f'PC1 ({ev1:.1f}%)')
    ax.set_ylabel(f'PC2 ({ev2:.1f}%)')
    ax.set_title(f'{tech} {line}  ({note})', fontweight='bold')
    ax.legend(fontsize=9, markerscale=2)

fig.suptitle('PC1 vs PC2 coloured by condition (no labels used for PCA)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

Each dot on the graph is one cell, colored in red for hypoxia and blue for normoxia. PCA never sees the labels of the cells, so if the colors separate, it is because of their biology. 

**SmartSeq MCF7:** There's a separation of two distinct clouds along the PC1 axis, effectively making it the hypoxia/normoxia axis.

**SmartSeq HCC1807:** Some separation can be seen but there's more overlap than MCF7. HCC1807 has a different baseline expression profile, but this could also be due to the shorter time it was kept under conditions compare to MCF7.

**DropSeq:** There's a clear heavy overlap, and PC1 explains less that 1%. This is again due to dropout, which makes cells look similar, though a slight directional trend can still be seen.

Overall, even without labels, PCA recovers the two conditions, so hypoxia is the main source of variation.

### 5.5 Clustering

Clustering groups cells by similarity in their gene expressions without ever seeing the labels, to see if the two biological conditions separate naturally. We do this in a PCA-reduced space rather than the full 3000-gene space, namely the top 10 PCs, because no dataset reaches anywhere near 80% variance even with the top 30 PCs. Two complementary methods are used:

**K-means:** assigns cells to K clusters minimising the within-cluster variance

**Hierarchical (Ward):** builds a tree of merges using Ward linkage

We use two complementary methods to act like a cross-check: if both agree, the result is more reliable.

In [ ]:
X_pca = {}
for (tech, line), scores in scores_dict.items():
    X_pca[(tech, line)] = scores[:, :10]
    var_explained = pca_dict[(tech, line)].explained_variance_ratio_[:10].sum() * 100
    print(f"{tech:10s}{line:8s}: using top 10 PCs -> {var_explained:.1f}% variance")

### 5.5.1 Silhouette analysis: choosing the best K for K-means

K-means requires us to choose the K number of clusters upfront, so we perform silhouette analysis to help us pick the best one. It measures how well each cell fits into its own cluster compared to the next nearest cluster. The range is between -1 (wrong cluster) to +1 (perfectly assigned), and we pick the K with the highest score.

In [ ]:
K_RANGE   = range(2, 8)
SIL_CAP   = 3000
sil_results = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (tech, line) in zip(axes, X_pca.keys()):
    X_cl = X_pca[(tech, line)]
    scores = []

    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_cl)

        if X_cl.shape[0] > SIL_CAP:
            idx_s = RNG.choice(X_cl.shape[0], SIL_CAP, replace=False)
            sil = silhouette_score(X_cl[idx_s], labels[idx_s])
        else:
            sil = silhouette_score(X_cl, labels)
        scores.append(sil)

    sil_results[(tech, line)] = scores
    best_k = list(K_RANGE)[np.argmax(scores)]

    ax.plot(list(K_RANGE), scores, marker='o', lw=2)
    ax.axvline(best_k, color='red', ls='--', lw=1.3, label=f'Best K={best_k}')
    ax.set_xlabel('Number of clusters K')
    ax.set_ylabel('Silhouette score')
    ax.set_title(f'{tech} · {line}', fontweight='bold')
    ax.set_xticks(list(K_RANGE))
    ax.legend(fontsize=9)

fig.suptitle('K-means: silhouette score vs K (higher = better separation)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


#### 5.5.2 K-means clusters visualised in PCA space

The same PC1 vs PC2 scatter as before, but now the cells are colored by K-means cluster assignment instead of condition. This allows us to visually check whether the clusters K-means found actually separate in the PCA space and whether they match the condition split.

In [ ]:
km_labels = {}
best_k_dict = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for ax, ((tech, line), X_cl) in zip(axes, X_pca.items()):
    pca = pca_dict[(tech, line)]
    scores = scores_dict[(tech, line)]
    conds = cond_dict[(tech, line)]

    best_k = list(K_RANGE)[np.argmax(sil_results[(tech, line)])]
    best_k_dict[(tech, line)] = best_k

    km = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cl)
    km_labels[(tech, line)] = labels

    if scores.shape[0] > MAX_SCATTER:
        idx = RNG.choice(scores.shape[0], MAX_SCATTER, replace=False)
        s_plot = scores[idx, :2]
        l_plot = labels[idx]
        note = f'n={len(idx)} sampled'
    else:
        s_plot, l_plot = scores[:, :2], labels
        note = f'n={scores.shape[0]}'

    palette = plt.cm.tab10.colors
    for cl in range(best_k):
        mask = l_plot == cl
        ax.scatter(s_plot[mask, 0], s_plot[mask, 1], color=palette[cl], alpha=0.55, s=18, label=f'Cluster {cl}', rasterized=True)

    ev1 = pca.explained_variance_ratio_[0] * 100
    ev2 = pca.explained_variance_ratio_[1] * 100
    ax.set_xlabel(f'PC1 ({ev1:.1f}%)')
    ax.set_ylabel(f'PC2 ({ev2:.1f}%)')
    ax.set_title(f'{tech} {line}  K={best_k}  ({note})', fontweight='bold')
    ax.legend(fontsize=8, markerscale=2)

fig.suptitle('K-means clusters in PC space (best K per dataset)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In the silhouette score vs K graph, the red line marks the best K:

**SmartSeq MCF7 and HCC1806:** Both peak sharply at K=2, meaning the data strongly prefers two clusters, which matches the two conditions. The scores are around 0.36-0.37, which indicates moderately good separation of the clusters.

**DropSeq HCC1806:** This also peaks at K=2 but with a much lower score of around 0.2 and a flatter curve, which implies two groups were found but not well-separated.

**DropSeq MCF7:** This is the only exception, where it peaks at K=3, which is likely noise creating an artificial third cluster rather than a genuine biologically different condition.

Generally, 3 out of 4 datasets independently chose K=2, matching the two conditions without ever seeing the labels. The chosen K's also coincide with the K-means cluster scatter in the PCA space.

#### 5.5.3 Cluster vs condition contingency table

How much do the unsupervised K-means clusters align with the Hypoxia/Normoxia labels?  

In [ ]:
for (tech, line), labels in km_labels.items():
    conds = cond_dict[(tech, line)]
    ct = pd.crosstab(pd.Series(labels, name='K-means cluster'), pd.Series(conds, name='Condition'))
    ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    print(f"\n{'-'*55}")
    print(f"{tech} {line}(K={best_k_dict[(tech, line)]})")
    print(ct)
    print("Row % (how much of each cluster is Hypoxia vs Normoxia):")
    print(ct_pct)

**SmartSeq MCF7 (K=2):** Only 1 out of 250 cells is misassigned in cluster 1, making it near-perfect.

**SmartSeq HCC1806 (K=2):** Cluster 0 is 68% hypoxia 32% normoxia, and cluster 1 is balanced with 51% hypoxia and 49% normoxia, so it is much weaker.

**DropSeq HCC1806 (K=3):** Cluster 1 is mostly hypoxia (97%) but clusterd 0 and 2 are mostly normoxia (78% and 86%), meaning normoxia split into two groups rather than it being a genuine third cluster.

**DropSeq MCF7 (K=2):** Both clusters are quite mixed, with 68% hypoxia 32% normoxia in cluster 0 and 48% hypoxia 52% normoxia in cluster 1, the noise overhwelms the signal.

It seems like clustering quality degrades with data quality, having SmartSeq MCF7 being nearly perfect, SmartSeq HCC1806 partially, and DropSeq essentially random.

#### 5.5.4 Hierarchical clustering (Ward linkage) + dendrograms

Hierarchical clustering builds a tree of cell merges bottom-up, where every cell starts as its own cluster, then progressively merges the most similar pairs. Ward linkage minimises total within-cluster variance at each merge. Since DropSeq has too many cells to plot fully, we subsample 300 cells for visualisation only. 

In [ ]:
DENDRO_CAP = 300
hc_labels = {}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, ((tech, line), X_cl) in zip(axes, X_pca.items()):
    conds = cond_dict[(tech, line)]
    best_k = best_k_dict[(tech, line)]

    if X_cl.shape[0] > DENDRO_CAP:
        idx = RNG.choice(X_cl.shape[0], DENDRO_CAP, replace=False)
        X_den = X_cl[idx]
        c_den = conds[idx]
        note = f'n={len(idx)} sampled'
    else:
        X_den, c_den = X_cl, conds
        note = f'n={X_cl.shape[0]}'

    Z = linkage(X_den, method='ward')
    color_thresh = 0.5 * Z[-1, 2] #cutoff at 50% of the max merge distance

    leaf_labels = ['H' if c == 'Hypoxia' else 'N' for c in c_den]
    dendrogram(Z, ax=ax, labels=leaf_labels, color_threshold=color_thresh, leaf_rotation=90, leaf_font_size=5, above_threshold_color='grey')

    ax.axhline(color_thresh, color='red', ls='--', lw=1.3, label=f'Cut @ {color_thresh:.1f}')
    ax.set_title(f'{tech} {line} Ward hierarchical({note})', fontweight='bold')
    ax.set_xlabel('Cell  (H=Hypoxia, N=Normoxia)')
    ax.set_ylabel('Ward distance')
    ax.legend(fontsize=8)

    hc = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
    hc_labels[(tech, line)] = hc.fit_predict(X_cl)

fig.suptitle('Hierarchical clustering dendrograms (Ward linkage)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

Each cell starts as a leaf at the bottom, and the algorithm merges them upwards. The y-axis is the Ward distance, or the height of a merge, which represents how different the two groups being joined were. The higher the final merge, the more different the last two groups were. The red dashed line is a cutoff at 50% of the maximum merge distance, that helps us visualize how different the last two groups were: if the two main branches were formed under this line, the data has an obvious two group structure.

**SmartSeq MCF7:** The only dataset where two groups were formed under the red line, implying these two are genuinely distinct, clearly supporting K=2.
**SmartSeq HCC1806 and DropSeq:** There is no clear two branch structure, having many small branches at similar heights, and the red line cutting through a messy region. This supports the weaker separation seen in PCA and clustering.

### 5.5.5 Silhouette score summary table: K-means vs Hierarchical

In [ ]:
summary_rows = []

for (tech, line), X_cl in X_pca.items():
    best_k = best_k_dict[(tech, line)]

    if X_cl.shape[0] > SIL_CAP:
        idx_s = RNG.choice(X_cl.shape[0], SIL_CAP, replace=False)
    else:
        idx_s = np.arange(X_cl.shape[0])

    sil_km = silhouette_score(X_cl[idx_s], km_labels[(tech, line)][idx_s])
    sil_hc = silhouette_score(X_cl[idx_s], hc_labels[(tech, line)][idx_s])

    summary_rows.append({
        'Technology': tech, 'Cell line': line, 'K': best_k,
        'Silhouette K-means': round(sil_km, 4),
        'Silhouette Hierarchical': round(sil_hc, 4),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

The summary table compares the silhouette scores for both methods side by side across all 4 datasets, which will help us confirm whether K-means and hierarchical agree and quantifies the quality of separation.

**SmartSeq** are above the 0.3 threshold for meaningful cluster structure, both methods agree on K=2.

**DropSeq** scores are weaker but not zero, so the biological signal exists but is partially drowned out by dropout noise.

K-means and hierarchical give similar scores across all four datasets. Two independent methods givivng the same score makes the result more reliable and less likely to be a coincidence or a quirk of one method. The bottom line is, cluster quality follows data quality, with Smart Seq separating more cleanly and DropSeq less so.

## 5.6. Cross-dataset comparison

We compare the unsupervised structure along two axes:

**Axis 1: Cell line:** MCF7 vs HCC1806 within the same sequencing technology.  
**Axis 2: Technology:** SmartSeq vs DropSeq for the same cell line.

Since each dataset has its own PCA space (fitted on its own genes), absolute coordinates differ, but we can compare the degree of separation of the two conditions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, tech in zip(axes, ['SmartSeq', 'DropSeq']):
 for line, marker, color in [('MCF7', 'o', '#2196F3'), ('HCC1806', '^', '#FF5722')]:
 scores = scores_dict[(tech, line)]
 conds = cond_dict[(tech, line)]

 if scores.shape[0] > MAX_SCATTER // 2:
 idx = RNG.choice(scores.shape[0], MAX_SCATTER // 2, replace=False)
 s_plot = scores[idx, :2]
 c_plot = conds[idx]
 else:
 s_plot, c_plot = scores[:, :2], conds

 for cond, alpha in [('Hypoxia', 0.7), ('Normoxia', 0.35)]:
 mask = c_plot == cond
 lbl = f'{line}' if cond == 'Hypoxia' else None
 ax.scatter(s_plot[mask, 0], s_plot[mask, 1], color=color, marker=marker, alpha=alpha, s=18, label=lbl, rasterized=True)

 ax.set_xlabel('PC1')
 ax.set_ylabel('PC2')
 ax.set_title(f'{tech}: MCF7 vs HCC1806\n(darker = Hypoxia, lighter = Normoxia)', fontweight='bold')
 ax.legend(fontsize=9, markerscale=2)

fig.suptitle('Cross-dataset comparison - Cell line axis', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

**Blue circles:** MCF7, **Red Triangles:** HCC1807, darker: Hypoxia, lighter: Normoxia

**SmartSeq:** The two cell lines occupy somewhat different areas of the PCA space, because of their different expression profiles. But, within each cell line, lighter/darker shade separation is visible, confirming there is a hypoxia signal in both.

**DropSeq:** Both cell lines are heavily mixed and overlapping, the noise overwhelms any cell-line separation.

The key result from this is that cell lines have different enough baseline expressions that a classifier trained on one might not transfer to the other.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, line in zip(axes, ['MCF7', 'HCC1806']):
 for tech, marker, color in [('SmartSeq', 'o', '#9C27B0'), ('DropSeq', 's', '#4CAF50')]:
 scores = scores_dict[(tech, line)]
 conds = cond_dict[(tech, line)]

 if scores.shape[0] > MAX_SCATTER // 2:
 idx = RNG.choice(scores.shape[0], MAX_SCATTER // 2, replace=False)
 s_plot = scores[idx, :2]
 c_plot = conds[idx]
 else:
 s_plot, c_plot = scores[:, :2], conds

 for cond, alpha in [('Hypoxia', 0.7), ('Normoxia', 0.35)]:
 mask = c_plot == cond
 lbl = f'{tech}' if cond == 'Hypoxia' else None
 ax.scatter(s_plot[mask, 0], s_plot[mask, 1], color=color, marker=marker, alpha=alpha, s=18, label=lbl, rasterized=True)

 ax.set_xlabel('PC1')
 ax.set_ylabel('PC2')
 ax.set_title(f'{line}: SmartSeq vs DropSeq\n(darker = Hypoxia, lighter = Normoxia)', fontweight='bold')
 ax.legend(fontsize=9, markerscale=2)

fig.suptitle('Cross-dataset comparison - Technology axis', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

**Purple:** SmartSeq, **Green:*** DropSeq, darker: Hypoxia, lighter: Normoxia

**SmartSeq:** The cells spread widely on the graph but with a clear dark/light separation. This implies deep sequencing produces distinct profiles PCA can separate cleanly.

**DropSeq:** The cells are compressed into a tight cluster, which tracks with the shallow sequencing dropout that produces flat and similar looking profiles.

Still, both technologies show the same directional trend for hypoxia and normoxia, meaning that regardless of technology, the biological trend is still captured. 

### 5.7. Unsupervised Summary

|Step|What we did|Why|
|----|-----------|---|
|Log Transform|Applied log2(x+1) to all counts|Compresses dynamic range, stabilises variance|
|Z-score|Scaled each gene to mean=0, std=1|Prevents highly expressed genes from dominating PCA|
|Correlation heatmaps|Computed cell-cell Pearson correlations|Checks if hypoxia drives cell to cell differences|
|Outlier flagging|Flagged cells >3 SD below mean correlation|Identifies low quality or anomalous cells|
|PCA|Reduced 3000 genes to top 10 PCs|Captures main biological structure, reduces noise|
|K-means clustering|Clustered cells in PCA space, K chosen by silhouette|Tests if two biological groups emerge without labels|
|Hierarchical clustering|Ward linkage tree on PCA space|Cross-checks K-means findings|
|Cross-dataset comparison|Overlaid PCA plots by cell line adn technology|Quantifies how structure differs across datasets|

### Results flow

| |SmartSeq MCF7|SmartSeq HCC1806|DropSeq MCF7|DropSeq HCC1806|
|-----|-----|-----|-----|-----|
|Outliers found|0|4(Hypoxia)|0|0|
|PC1 variance|10.6%|4.5%|0.7%|0.4%|
|Variance (top 10 PCs)|25.0%|19.8%|1.8%|1.5%|
|Best K|2|2|3|2|
|Silhouette (K-means)|0.356|0.368|0.259|0.206|
|Silhouette (Hierarchical)|0.350|0.459|0.206|0.167|
|Cluster-condition match|Near-perfect|Partial|Weak|Weak|

## 6. Supervised learning

### 6.1 Introduction

This section trains and evaluates supervised classifiers for hypoxia detection on two DropSeq datasets: **MCF7** and **HCC1806**. We will be running both cell lines in parallel at each step so results can be compared directly.

Supervised classification focuses *only* on DropSeq, as the unsupervised analysis demonstrated that SmartSeq data separates almost perfectly without labels (SmartSeq MCF7 K-means: 249/250 correctly grouped, as shown in section 4.5.3), making supervised classification on SmartSeq an uninteresting baseline.

The two lines were profiled under identical conditions (Hypoxia / Normoxia) but differ in subtype, baseline expression, and hypoxia exposure time (72h for MCF7, 24h for HCC1806). From the unsupervised analysis we already know that MCF7 shows a cleaner hypoxia signal than HCC1806 in DropSeq data. The aim here is to quantify that difference in a supervised setting and check whether it holds across all four classifiers.

**Pipeline per cell line (run in parallel):**
1. Load DropSeq train / anonymous test matrices
2. Extract Hypoxia / Normoxia labels from cell barcodes
3. Feature selection: variance threshold -> top-500 mutual information genes
4. 5-fold stratified cross-validation (LR, SVM-RBF, RF, GBM)
5. Train on the full training set and serialise models
6. Predict on the anonymous test set
7. Random Forest feature importance
8. Cross-line comparison of CV accuracy and top genes


The four classifiers were chosen to cover a range of model families: **Logistic Regression** as a linear baseline, **SVM-RBF** for non-linear separation via the kernel trick (well-suited to high-dimensional sparse data), and **Random Forest** and **Gradient Boosting** as ensemble tree methods with different bias-variance tradeoffs. RF reduces variance through bagging, GBM reduces bias through sequential boosting.

In [ ]:
LINES = ['MCF7', 'HCC1806']
TECHS = ['DropSeq', 'SmartSeq']
KEYS = [('DropSeq', 'MCF7'), ('DropSeq', 'HCC1806'), ('SmartSeq', 'MCF7'), ('SmartSeq', 'HCC1806')]

N_TOP_GENES = 500
RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### 6.2 Load the necessary files

Each matrix is stored **genes × cells**; we transpose to get the standard **cells × genes** format.

- `train_raw`: labelled cells, where barcode encodes condition (example: `AAAAACCTATCG_Normoxia`)
- `test_raw`: anonymous cells, there is no label, predictions saved at the end


In [ ]:
train_raw = {
    ('DropSeq',  'MCF7'):    load_matrix('data/DropSeq/MCF7_Filtered_Normalised_3000_Data_train.txt').T,
    ('DropSeq',  'HCC1806'): load_matrix('data/DropSeq/HCC1806_Filtered_Normalised_3000_Data_train.txt').T,
    ('SmartSeq', 'MCF7'):    load_matrix('data/SmartSeq/MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt').T,
    ('SmartSeq', 'HCC1806'): load_matrix('data/SmartSeq/HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt').T,
}
test_raw = {
    ('DropSeq',  'MCF7'):    load_matrix('data/DropSeq/MCF7_Filtered_Normalised_3000_Data_test_anonim.txt').T,
    ('DropSeq',  'HCC1806'): load_matrix('data/DropSeq/HCC1806_Filtered_Normalised_3000_Data_test_anonim.txt').T,
    ('SmartSeq', 'MCF7'):    load_matrix('data/SmartSeq/MCF7_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt').T,
    ('SmartSeq', 'HCC1806'): load_matrix('data/SmartSeq/HCC1806_SmartS_Filtered_Normalised_3000_Data_test_anonim.txt').T,
}


for key in KEYS:
    print(f"{key}  train: {train_raw[key].shape}  test: {test_raw[key].shape}")
    print(f"sample barcodes: {train_raw[key].index[:3].tolist()}")

### 6.3 Extract Labels

Labels are the last `_`-delimited token in each training barcode. The anonymous test set carries no label.

In [ ]:
y_train = {}
x_train = {}
x_test  = {}

for key in KEYS:
    y_train[key] = np.array([get_condition(c) for c in train_raw[key].index])
    x_train[key] = train_raw[key].values
    x_test[key]  = test_raw[key].values
    dist = pd.Series(y_train[key]).value_counts().to_dict()
    print(f"{key}  label distribution: {dist}  |  test cells: {x_test[key].shape[0]}")

### 6.4 Feature selection

Two-stage dimensionality reduction, **fit on train only** to avoid data leakage:

1. **Variance threshold** (`threshold=0.01`): remove near-constant genes.
2. **Mutual information** (`mutual_info_classif`): keep the top `N_TOP_GENES = 500` genes most informative about the Hypoxia/Normoxia label.

Variance threshold is applied first because it is computationally cheap and removes genes with near-zero variance that carry no information, so running MI on them would waste computation and add noise to the ranking. Stratified k-fold is used rather than standard k-fold because HCC1806 has mild class imbalance (126 vs 117) and stratification guarantees each fold preserves the original class ratio, preventing folds where one class is underrepresented.

The same fitted transformers are applied to the test set without refitting.



In [ ]:
var_filter   = {}
x_train_var  = {}
x_test_var   = {}
mi_scores    = {}
top_idx      = {}
top_genes    = {}
x_train_top  = {}
x_test_top   = {}

for key in KEYS:
    # variance filter
    vf = VarianceThreshold(threshold=0.01)
    x_train_var[key] = vf.fit_transform(x_train[key])
    x_test_var[key]  = vf.transform(x_test[key])
    var_filter[key]  = vf
    print(f"{key[0]:10s} {key[1]:8s}  genes before: {x_train[key].shape[1]}  after variance filter: {x_train_var[key].shape[1]}")

    # mutual information
    mi = mutual_info_classif(x_train_var[key], y_train[key], random_state=RANDOM_STATE)
    idx = np.argsort(mi)[::-1][:N_TOP_GENES]
    mi_scores[key] = mi
    top_idx[key]   = idx
    x_train_top[key] = x_train_var[key][:, idx]
    x_test_top[key]  = x_test_var[key][:, idx]

    genes_after_var = train_raw[key].columns[vf.get_support()]
    top_genes[key] = genes_after_var[idx]
    print(f"{key[0]:10s} {key[1]:8s}  genes after MI selection: {x_train_top[key].shape[1]}")

#### 6.4.1 Top 20 genes by mutual information: MCF7 vs HCC1806

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, key in zip(axes.flat, KEYS):
    sns.barplot(x=mi_scores[key][top_idx[key]][:20], y=top_genes[key][:20], palette='viridis', ax=ax)
    ax.set_title(f'{key[0]} {key[1]}: top 20 genes by MI')
    ax.set_xlabel('MI score')
plt.suptitle('Top informative genes per dataset', fontweight='bold')
plt.tight_layout()
plt.show()

### 6.5 Cross-validation

5-fold stratified cross-validation on `x_train_top` for all four classifiers, run independently on each cell line.


In [ ]:
models_def = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'SVM':                SVC(kernel='rbf', random_state=RANDOM_STATE),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

cv_results = {}
for key in KEYS:
    cv_results[key] = {}
    print(f"\n--- {key[0]} {key[1]} ---")
    for name, model in models_def.items():
        scores = cross_val_score(model, x_train_top[key], y_train[key], cv=cv, scoring='accuracy')
        cv_results[key][name] = scores
        print(f"  {name:25s}  mean={scores.mean():.4f}  std={scores.std():.4f}")


#### 6.5.1 CV accuracy distribution side by side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
for ax, key in zip(axes.flat, KEYS):
    sns.boxplot(data=pd.DataFrame(cv_results[key]), palette='Set2', ax=ax)
    ax.set_title(f'{key[0]} {key[1]}: 5-fold CV accuracy')
    ax.set_ylabel('Accuracy')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Cross-validation accuracy across all datasets', fontweight='bold')
plt.tight_layout()
plt.show()


SVM-RBF achieves the highest mean CV accuracy on all four datasets, confirming that the hypoxia signal is not linearly separable and benefits from the kernel trick. The SmartSeq datasets score near-perfectly (expected given the clean separation seen in the unsupervised analysis), while DropSeq scores are lower, reflecting higher sparsity and noise. Within DropSeq, MCF7 consistently outperforms HCC1806 by ~3pp, consistent with its longer 72h hypoxia exposure producing a stronger and more developed transcriptional response. RF and GBM perform similarly to each other but below SVM, suggesting that the decision boundaries in this feature space are better captured by a kernel method than by axis-aligned tree splits.

#### 6.5.2. Mean CV accuracy: direct comparison table

In [ ]:
rows = []
for key in KEYS:
    for name, scores in cv_results[key].items():
        rows.append({'Technology': key[0], 'Cell line': key[1], 'Model': name,
                     'Mean accuracy': round(scores.mean(), 4), 'Std': round(scores.std(), 4)})
cv_table = pd.DataFrame(rows).pivot_table(index='Model', columns=['Technology', 'Cell line'], values='Mean accuracy')
print(cv_table.to_string())


### 6.6. Training on full set

Each of the 16 models (4 datasets × 4 classifiers) is trained independently on its own dataset's feature-selected training data. For each dataset, a fresh model instance is created via `type(model)(**model.get_params())` to avoid any state leaking between fits, then trained on `x_train_top[key]` - the 500 MI genes selected specifically for that dataset. No information is shared across datasets during training.


In [ ]:
param_grids = {
    'SVM': {
        'C':     [1, 10, 50, 100],
        'gamma': ['scale', 0.001, 0.01],
    },
    'LogisticRegression': {
        'C':       [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver':  ['liblinear'],
    },
}

os.makedirs('outputs/models', exist_ok=True)
trained_models = {}
for key in KEYS:
    trained_models[key] = {}
    for name, model in models_def.items():
        if key == ('DropSeq', 'HCC1806') and name in param_grids:
            gs = GridSearchCV(model, param_grids[name], cv=cv, scoring='accuracy', n_jobs=-1)
            gs.fit(x_train_top[key], y_train[key])
            m = gs.best_estimator_
            print(f'{key[0]:10s} {key[1]:8s}  {name}: best params={gs.best_params_}  score={gs.best_score_:.4f}')
        else:
            m = type(model)(**model.get_params())
            m.fit(x_train_top[key], y_train[key])
            print(f'{key[0]:10s} {key[1]:8s}  {name}: trained')
        trained_models[key][name] = m
        path = f'outputs/models/{key[0]}_{key[1]}_{name}.pkl'
        with open(path, 'wb') as f:
            pickle.dump(m, f)

### 6.7. Predict on an anonymous test set

Best model selected by highest mean CV accuracy (SVM-RBF in both lines based on prior runs).

In [ ]:
BEST_MODEL = {
    ('DropSeq',  'MCF7'):    'SVM',
    ('DropSeq',  'HCC1806'): 'SVM',
    ('SmartSeq', 'MCF7'):    'LogisticRegression',
    ('SmartSeq', 'HCC1806'): 'LogisticRegression',
}

for key in KEYS:
    best = trained_models[key][BEST_MODEL[key]]
    y_pred = best.predict(x_test_top[key])
    pred_df = pd.DataFrame({'cell': test_raw[key].index, 'predicted_label': y_pred})
    out_path = f'outputs/{key[0]}_{key[1]}_predictions.csv'
    pred_df.to_csv(out_path, index=False)
    dist = pd.Series(y_pred).value_counts().to_dict()
    print(f"{key[0]:10s} {key[1]:8s}  predictions: {dist}  | saved: {out_path}")


### 6.8 Random Forest feature importance

RF `feature_importances_` (mean decrease in impurity) is complementary to the MI scores from 5.5.: MI measures marginal relevance to the label; RF importance reflects contribution within the trained ensemble.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, key in zip(axes.flat, KEYS):
    rf = trained_models[key]['RandomForest']
    imp = pd.Series(rf.feature_importances_, index=top_genes[key]).sort_values(ascending=False)
    sns.barplot(x=imp.values[:20], y=imp.index[:20], palette='magma', ax=ax)
    ax.set_title(f'{key[0]} {key[1]}: top 20 genes by RF importance')
    ax.set_xlabel('Importance')
plt.suptitle('Random Forest feature importance across all datasets', fontweight='bold')
plt.tight_layout()
plt.show()


### 6.7 Cross-line comparison

#### 6.7.1 Top-gene overlap

How many of the top-500 MI genes are shared between the two cell lines? A large overlap suggests a shared hypoxia transcriptional programme; a small overlap suggests the two lines activate different gene sets.

In [ ]:
print("=== Within-technology gene overlap (MCF7 vs HCC1806) ===")
for tech in TECHS:
 s1 = set(top_genes[(tech, 'MCF7')])
 s2 = set(top_genes[(tech, 'HCC1806')])
 print(f"{tech}: shared={len(s1 & s2)}, MCF7 only={len(s1 - s2)}, HCC1806 only={len(s2 - s1)}")

print()
print("=== Cross-technology gene overlap (DropSeq vs SmartSeq) ===")
for line in LINES:
 s1 = set(top_genes[('DropSeq', line)])
 s2 = set(top_genes[('SmartSeq', line)])
 print(f"{line}: shared={len(s1 & s2)}, DropSeq only={len(s1 - s2)}, SmartSeq only={len(s2 - s1)}")

# Bar chart - within technology
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, tech in zip(axes, TECHS):
 s1 = set(top_genes[(tech, 'MCF7')])
 s2 = set(top_genes[(tech, 'HCC1806')])
 ax.bar(['MCF7 only', 'Shared', 'HCC1806 only'],
 [len(s1 - s2), len(s1 & s2), len(s2 - s1)],
 color=['#2196F3', '#9C27B0', '#FF5722'])
 ax.set_title(f'{tech}: MCF7 vs HCC1806 gene overlap')
 ax.set_ylabel('Genes')
plt.suptitle(f'Top-{N_TOP_GENES} MI gene overlap within technology', fontweight='bold')
plt.tight_layout()
plt.show()


#### 6.7.2 CV accuracy: MCF7 vs HCC1806 bar chart

In [ ]:
rows = []
for key in KEYS:
    for name, scores in cv_results[key].items():
        rows.append({'Dataset': f'{key[0]}\n{key[1]}', 'Model': name,
                     'Mean CV accuracy': scores.mean()})
df_plot = pd.DataFrame(rows)

plt.figure(figsize=(12, 5))
sns.barplot(data=df_plot, x='Model', y='Mean CV accuracy', hue='Dataset', palette='Set1')
plt.title('Mean 5-fold CV accuracy across all datasets', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### 6.8 Summary of results
### Feature selection
Both lines start from 3,000 HVGs. After variance filtering and top-500 MI selection, the surviving gene sets overlap partially but not completely, suggesting a shared core hypoxia programme alongside cell-line-specific responses.

### Cross-validation

| Model | DropSeq MCF7 | DropSeq HCC1806 | SmartSeq MCF7 | SmartSeq HCC1806 |
|---|---|---|---|---|
| Logistic Regression | 0.9767 | 0.9472 | **1.0000** | **0.9890** |
| SVM (RBF) | **0.9807** | **0.9541** | 0.9920 | 0.9781 |
| Random Forest | 0.9699 | 0.9318 | 0.9960 | 0.9890 |
| Gradient Boosting | 0.9703 | 0.9313 | 0.9760 | 0.9560 |

SVM-RBF is the best model on both DropSeq datasets; Logistic Regression matches or exceeds it on SmartSeq. Predictions are generated using the best model per dataset. SmartSeq accuracies approach ceiling, consistent with the near-perfect unsupervised separation seen in section 4. The persistent ~3pp gap between MCF7 and HCC1806 on DropSeq reflects the difference in hypoxia exposure time (72h vs 24h).


## 7. Generalization

This section tests whether the classifiers trained in section 5 generalize beyond their training conditions. Two types of generalization are evaluated:

1. **Cross-cell-line**: a model trained on MCF7 cells is applied to HCC1806 cells (and vice versa). Same technology (DropSeq), different biology.
2. **Cross-technology**: a model trained on DropSeq data is applied to SmartSeq data from the same cell line, and vice versa. Same biology, different sequencing platform.

### 7.1 Gene-set overlap summary

Before running models across conditions, it is useful to know how much the feature sets overlap. Low overlap means a model will receive mostly zero-filled inputs when applied to a different condition: the closer to chance the performance, the more this is a feature-coverage problem rather than a biology problem.

In [ ]:
DATA_DROPSEQ  = 'data/DropSeq'
DATA_SMARTSEQ = 'data/SmartSeq'

# Load the same train files used in section 5, with explicit paths
gen_data = {
    ('DropSeq',  'MCF7'):    load_matrix(f'{DATA_DROPSEQ}/MCF7_Filtered_Normalised_3000_Data_train.txt').T,
    ('DropSeq',  'HCC1806'): load_matrix(f'{DATA_DROPSEQ}/HCC1806_Filtered_Normalised_3000_Data_train.txt').T,
    ('SmartSeq', 'MCF7'):    load_matrix(f'{DATA_SMARTSEQ}/MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt').T,
    ('SmartSeq', 'HCC1806'): load_matrix(f'{DATA_SMARTSEQ}/HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt').T,
}
gen_labels = {
    key: np.array([get_condition(c) for c in df.index])
    for key, df in gen_data.items()
}
print('Loaded:', {k: gen_data[k].shape for k in KEYS})

In [ ]:
print('=== Within-technology gene overlap (MCF7 vs HCC1806) ===')
for tech in ['DropSeq', 'SmartSeq']:
    s1 = set(top_genes[(tech, 'MCF7')])
    s2 = set(top_genes[(tech, 'HCC1806')])
    print(f'{tech}    MCF7 vs HCC1806 : {len(s1 & s2)} shared  '
          f'({len(s1 - s2)} MCF7-only, {len(s2 - s1)} HCC1806-only)')

print()
print('=== Cross-technology gene overlap (DropSeq vs SmartSeq) ===')
for line in ['MCF7', 'HCC1806']:
    s1 = set(top_genes[('DropSeq', line)])
    s2 = set(top_genes[('SmartSeq', line)])
    print(f'{line:8s}  DropSeq vs SmartSeq : {len(s1 & s2)} shared  '
          f'({len(s1 - s2)} DropSeq-only, {len(s2 - s1)} SmartSeq-only)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: within-technology overlap (MCF7 vs HCC1806)
for ax, tech in zip(axes, ['DropSeq', 'SmartSeq']):
    s1 = set(top_genes[(tech, 'MCF7')])
    s2 = set(top_genes[(tech, 'HCC1806')])
    ax.bar(['MCF7 only', 'Shared', 'HCC1806 only'],
           [len(s1 - s2), len(s1 & s2), len(s2 - s1)],
           color=['#2196F3', '#9C27B0', '#FF5722'], edgecolor='white')
    ax.set_title(f'{tech}: top-500 gene overlap\nMCF7 vs HCC1806')
    ax.set_ylabel('Number of genes')
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

plt.suptitle('Cross-cell-line gene overlap (within technology)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/gen_gene_overlap_crosscell.png', dpi=150)
plt.show()

# Cross-technology overlap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, line in zip(axes, ['MCF7', 'HCC1806']):
    s1 = set(top_genes[('DropSeq', line)])
    s2 = set(top_genes[('SmartSeq', line)])
    ax.bar(['DropSeq only', 'Shared', 'SmartSeq only'],
           [len(s1 - s2), len(s1 & s2), len(s2 - s1)],
           color=['#FF9800', '#4CAF50', '#2196F3'], edgecolor='white')
    ax.set_title(f'{line}: top-500 gene overlap\nDropSeq vs SmartSeq')
    ax.set_ylabel('Number of genes')
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

plt.suptitle('Cross-technology gene overlap (within cell line)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/gen_gene_overlap_crosstech.png', dpi=150)
plt.show()

### 7.2 Helper functions

In [ ]:
def select_genes(df, gene_list):
    """
    Subset df columns to gene_list in training order.
    Genes absent from df are zero-filled (not detected / different platform).
    Built in a single concat to avoid DataFrame fragmentation.
    """
    present = [g for g in gene_list if g in df.columns]
    missing = [g for g in gene_list if g not in df.columns]
    parts = [df[present]]
    if missing:
        parts.append(pd.DataFrame(0.0, index=df.index, columns=missing))
    return pd.concat(parts, axis=1)[gene_list].values


def run_models(cell_line_models, X, y, scenario, train_tech, train_line, test_tech, test_line):
    rows = []
    for mname, model in cell_line_models.items():
        yp = model.predict(X)
        rows.append({
            'scenario':   scenario,
            'train_tech': train_tech,
            'train_line': train_line,
            'test_tech':  test_tech,
            'test_line':  test_line,
            'model':      mname,
            'acc':        round(accuracy_score(y, yp), 4),
            'bacc':       round(balanced_accuracy_score(y, yp), 4),
        })
    return rows


### 7.3 In-Distribution Baseline

Each model is re-applied to the data it was **trained on** (training set, not held-out). Scores here are inflated for models that overfit (especially Random Forest, which memorises training data). These are included only as a ceiling reference, not as valid generalisation estimates.

In [ ]:
results = []

for key in KEYS:
    X = select_genes(gen_data[key], top_genes[key])
    y = gen_labels[key]
    results += run_models(
        trained_models[key], X, y,
        scenario='in-dist',
        train_tech=key[0], train_line=key[1],
        test_tech=key[0],  test_line=key[1],
    )

pd.DataFrame(results).sort_values(['train_tech', 'train_line', 'model'])


### 7.4 Cross-Cell-Line Generalization

DropSeq-trained models are applied to the other cell line's DropSeq data. The two cell lines share only **105** of their top-500 MI genes (DropSeq). Genes absent from the target data are zero-filled.

In [ ]:
for train_line, test_line in [('MCF7', 'HCC1806'), ('HCC1806', 'MCF7')]:
    key_train = ('DropSeq', train_line)
    key_test  = ('DropSeq', test_line)
    df = gen_data[key_test]
    y  = gen_labels[key_test]
    n_present = sum(1 for g in top_genes[key_train] if g in df.columns)
    print(f'DropSeq {train_line} → {test_line}: {n_present}/500 training genes present in target')
    X = select_genes(df, top_genes[key_train])
    results += run_models(
        trained_models[key_train], X, y,
        scenario='cross-cell',
        train_tech='DropSeq', train_line=train_line,
        test_tech='DropSeq',  test_line=test_line,
    )

pd.DataFrame([r for r in results if r['scenario'] == 'cross-cell']).sort_values(['train_line', 'model'])


In [ ]:
cc_df = pd.DataFrame([r for r in results if r['scenario'] == 'cross-cell'])
cc_pivot = cc_df.pivot_table(
    values='bacc',
    index='model',
    columns=['train_line', 'test_line'],
)
# Also add in-dist for reference
id_df = pd.DataFrame([r for r in results if r['scenario'] == 'in-dist' and r['train_tech'] == 'DropSeq'])
id_pivot = id_df.pivot_table(values='bacc', index='model', columns='train_line')
id_pivot.columns = pd.MultiIndex.from_tuples([(c, c + ' (in-dist)') for c in id_pivot.columns])

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(cc_df['model'].unique()))
models_sorted = sorted(cc_df['model'].unique())
w = 0.2
colors = {'MCF7→HCC1806': '#e76f51', 'HCC1806→MCF7': '#2a9d8f',
          'MCF7 in-dist': '#e9c46a', 'HCC1806 in-dist': '#264653'}

for i, (direction, col) in enumerate([
    ('MCF7→HCC1806',  ('MCF7', 'HCC1806')),
    ('HCC1806→MCF7',  ('HCC1806', 'MCF7')),
]):
    vals = [cc_pivot.loc[m, col] if m in cc_pivot.index else 0 for m in models_sorted]
    ax.bar(x + i*w, vals, width=w, label=direction, color=colors[direction], edgecolor='white')

ax.axhline(0.5, linestyle='--', linewidth=0.8, color='grey', label='chance')
ax.set_xticks(x + w/2)
ax.set_xticklabels(models_sorted, rotation=15, ha='right')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Cross-cell-line generalization: balanced accuracy by direction and model')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('outputs/figures/gen_crosscell_bars.png', dpi=150)
plt.show()

### 7.5 Cross-Technology Generalization

Models trained on one sequencing technology are tested on the same cell line measured with the other technology. We test both directions:
- **DropSeq → SmartSeq**: DropSeq-trained model applied to SmartSeq data
- **SmartSeq → DropSeq**: SmartSeq-trained model applied to DropSeq data

The cross-technology gene overlap is very low (39 genes for MCF7, 48 for HCC1806) because mutual information feature selection was run independently on each dataset, and the two technologies measure slightly different gene sets with different noise profiles.

In [ ]:
for line in ['MCF7', 'HCC1806']:
    for train_tech, test_tech in [('DropSeq', 'SmartSeq'), ('SmartSeq', 'DropSeq')]:
        key_train = (train_tech, line)
        key_test  = (test_tech,  line)
        df = gen_data[key_test]
        y  = gen_labels[key_test]
        n_present = sum(1 for g in top_genes[key_train] if g in df.columns)
        print(f'{line} {train_tech} → {test_tech}: {n_present}/500 training genes present')
        X = select_genes(df, top_genes[key_train])
        results += run_models(
            trained_models[key_train], X, y,
            scenario='cross-tech',
            train_tech=train_tech, train_line=line,
            test_tech=test_tech,   test_line=line,
        )

pd.DataFrame([r for r in results if r['scenario'] == 'cross-tech']).sort_values(['train_line', 'train_tech', 'model'])


In [ ]:
ct_df = pd.DataFrame([r for r in results if r['scenario'] == 'cross-tech'])
ct_df['direction'] = ct_df['train_tech'] + '→' + ct_df['test_tech'] + ' ' + ct_df['train_line']

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, line in zip(axes, ['MCF7', 'HCC1806']):
    sub = ct_df[ct_df['train_line'] == line]
    pivot = sub.pivot_table(values='bacc', index='model', columns='direction')
    pivot = pivot.reindex(sorted(pivot.index))
    x = np.arange(len(pivot.index))
    w = 0.35
    dirs = sorted(pivot.columns)
    col_map = ['#e76f51', '#2a9d8f']
    for i, (d, c) in enumerate(zip(dirs, col_map)):
        ax.bar(x + i*w, pivot[d].values, width=w, label=d, color=c, edgecolor='white')
    ax.axhline(0.5, linestyle='--', linewidth=0.8, color='grey', label='chance')
    ax.set_xticks(x + w/2)
    ax.set_xticklabels(pivot.index, rotation=15, ha='right')
    ax.set_title(f'{line}: cross-technology balanced accuracy')
    ax.set_ylabel('Balanced accuracy')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)

plt.suptitle('Cross-technology generalization by model and direction', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/gen_crosstech_bars.png', dpi=150)
plt.show()

### 7.6 Summary and Visualization

In [ ]:
results_df = pd.DataFrame(results)

# For in-dist, average over the 4 datasets; for cross-cell/cross-tech average over directions
pivot = results_df.pivot_table(
    values='bacc',
    index='model',
    columns='scenario',
    aggfunc='mean',
)[['in-dist', 'cross-cell', 'cross-tech']]

fig, ax = plt.subplots(figsize=(9, 4))
palette = ['#2a9d8f', '#e76f51', '#e9c46a']
pivot.plot(kind='bar', ax=ax, color=palette, edgecolor='white', width=0.7)
ax.axhline(0.5, linestyle='--', linewidth=0.8, color='grey', label='chance (0.5)')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Generalization: mean balanced accuracy by scenario and model')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.legend(title='Scenario')
ax.set_ylim(0, 1.08)
plt.tight_layout()
plt.savefig('outputs/figures/generalization_summary.png', dpi=150)
plt.show()

print(pivot.round(3).to_string())


In [ ]:
# Performance drop relative to in-distribution, broken down by direction
in_dist_mean = results_df[results_df['scenario'] == 'in-dist'].groupby('model')['bacc'].mean()
cross = results_df[results_df['scenario'] != 'in-dist'].copy()
cross['in_dist_bacc'] = cross['model'].map(in_dist_mean)
cross['drop'] = (cross['in_dist_bacc'] - cross['bacc']).round(3)

print('Performance drop (in-dist bacc − cross-condition bacc):')
print(cross[['scenario', 'train_tech', 'train_line', 'test_tech', 'test_line', 'model', 'bacc', 'drop']]
      .sort_values(['scenario', 'train_line', 'model']).to_string(index=False))


### 7.7 Interpretation

#### In-distribution baseline

All models score very high on their own training data. SmartSeq models are uniformly at or near 1.00, reflecting the near-perfect hypoxia/normoxia separation that was already visible in the unsupervised analysis (section 4). DropSeq models range from 0.94 - 1.00, with Random Forest hitting 1.00 on both lines,a sign of memorisation rather than learning. SVM is the most accurate model that does not perfectly overfit, with balanced accuracy 0.989 (MCF7) and 0.981 (HCC1806) on the training set.

#### Cross-cell-line generalization

Performance drops substantially in both directions, but the drop is asymmetric:

- **MCF7 → HCC1806**: balanced accuracy falls to 0.54 - 0.57 across all models - barely above chance (0.50). The 500 MCF7-selected genes carry almost no information about the hypoxia state in HCC1806. This most likely reflects the difference in hypoxia exposure time (MCF7: 72 h, HCC1806: 24 h): MCF7 has had time to mount a full transcriptional response, selecting genes that are strongly regulated in MCF7 but may not yet be activated in HCC1806 after only 24 h.

- **HCC1806 → MCF7**: noticeably better (balanced accuracy 0.64 - 0.80). SVM reaches 0.80, Logistic Regression 0.69. The HCC1806 gene signature partially overlaps the MCF7 hypoxia response - the genes activated early under hypoxia (HCC1806, 24 h) are a subset of those that remain active at 72 h (MCF7).

The low gene overlap between the two cell lines (105/500 for DropSeq, 93/500 for SmartSeq) is a contributing factor, but not the whole story: even among the 179 - 212 DropSeq genes that are present in the target cell line's data, the MCF7 signal is too cell-line-specific to generalise downward (shorter exposure).

#### Cross-technology generalization

This is the hardest test, and the results are mostly poor - but with a notable exception:

- **SVM, Random Forest, and Gradient Boosting collapse to exactly 0.50** in all four cross-technology conditions. These are tree-based and kernel methods that build complex, high-dimensional decision boundaries. When 88 - 92 % of input features are zero-filled (because only 39 - 48 of 500 training genes are present in the other technology's data), the decision boundary is essentially undefined and the models predict a constant class, landing at chance-level balanced accuracy.

- **Logistic Regression is the only model that generalises across technologies**, and only in one direction: **DropSeq → SmartSeq MCF7** achieves 0.956 balanced accuracy, and **SmartSeq → DropSeq MCF7** achieves 0.746. HCC1806 cross-technology performance is poor for all models including LR (0.51 - 0.62), suggesting the MCF7 result is not a general property of LR but of the specific genes that happen to survive the cross-technology gap for MCF7.

The reason LR generalises where tree/kernel methods do not is structural: LR learns a sparse linear combination of the available (non-zero) genes, and if even a few of those genes carry the hypoxia signal consistently across technologies, LR can exploit them. SVM and ensemble methods need more of the feature space to be populated to function.

The cross-technology gene overlap is very low (39 for MCF7, 48 for HCC1806) because mutual information feature selection was run independently on each technology, capturing technology-specific noise patterns alongside the true biological signal. A model designed for cross-technology robustness would select features on the **intersection** of the two gene panels before training.

#### Overall conclusion

Generalisation degrades substantially as soon as training conditions change. The classifiers learned a mixture of true biology and condition-specific artefacts. The best single-direction result is SVM HCC1806 → MCF7 DropSeq (0.80 balanced accuracy), suggesting partial conservation of the early hypoxia response. Cross-technology generalisation is almost exclusively achieved by Logistic Regression on MCF7, likely because a small number of strongly regulated genes survive the DropSeq/SmartSeq gap for that cell line. For a deployable classifier, future work should: (1) select features on a shared gene panel across technologies; (2) use held-out cross-validation rather than in-sample evaluation; and (3) test on genuinely unseen biological replicates.


## 8. Conclusion

This project explored hypoxia classification in single-cell RNA sequencing data across two breast cancer cell lines (MCF7, HCC1806) and two sequencing technologies (SmartSeq, DropSeq).

The exploratory and unsupervised analysis established that the hypoxia signal is real and strong, while canonical hypoxia markers show consistent fold-changes in both lines, and unsupervised clustering recovers the Hypoxia/Normoxia split with near-perfect accuracy on SmartSeq and reasonable accuracy on DropSeq. The gap between technologies is structural: SmartSeq captures a wider dynamic range with lower dropout, making the signal trivially separable, while DropSeq's higher sparsity makes classification genuinely challenging.

The supervised classifiers confirmed this picture quantitatively. SVM-RBF was the best-performing model across all four datasets, benefiting from the kernel trick in a high-dimensional sparse feature space. SmartSeq classifiers approached ceiling performance (~99%), while DropSeq classifiers settled around ~95% for MCF7 and ~93% for HCC1806: the gap reflecting MCF7's longer 72h hypoxia exposure and correspondingly stronger transcriptional response.

The generalization experiments revealed the limits of these models. Cross-cell-line transfer degrades substantially in both directions, with MCF7→HCC1806 barely above chance, indicating that the two lines activate overlapping but distinct gene programmes. Cross-technology transfer collapses almost entirely due to low feature overlap between DropSeq- and SmartSeq-selected gene panels (~80 - 100 shared genes out of 500). The one exception (Logistic Regression generalizing well on SmartSeq MCF7) suggests that simpler linear models are more robust when feature overlap is sparse, consistent with the bias-variance tradeoff: complex models overfit to technology-specific noise that does not transfer.

The central lesson is that a classifier trained on one dataset does not generalize reliably to another unless the feature space is shared. A natural extension would be to define a common gene panel across all four datasets before training, which would directly test whether the hypoxia signature is universal or technology- and cell-line-specific.